# Phase 1

In [ ]:
# ================================
# Phase 1 / Cell 1 — Project Setup (agnostic of model/approach)
# ================================

import os, random, numpy as np, torch
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

CONFIG = {
    # Repro & runtime
    "SEED": 42,
    "NUMERIC": {
        "torch_float32_matmul_precision": "high",  # 'highest' | 'high' | 'medium'
        "allow_tf32": True,
        "deterministic": True,
        "cudnn_benchmark": False,
    },

    # Data
    "DATA_DIR": "./data",
    "IMG_SIZE": (224, 224),
    "NUM_CLASSES": 2,
    "TRAIN_SAMPLES": 10_000,
    "VAL_SAMPLES": 2_000,
    # CelebA-specific convenience (used later; safe to keep here)
    "CELEBA_GENDER_ATTR_IDX": 20,  # 'Male' attribute index
    # Training knobs (kept here to avoid duplication later)
    "LR": 1e-3,
    "BATCH_SIZE": 64,
    "EPOCHS": 5,
}

# --- paths
os.makedirs(CONFIG["DATA_DIR"], exist_ok=True)

# --- reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int):
    s = CONFIG["SEED"] + worker_id
    random.seed(s)
    np.random.seed(s)

# --- torch runtime policy
def configure_torch_numeric(nc: dict):
    torch.set_float32_matmul_precision(nc.get("torch_float32_matmul_precision", "high"))
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = bool(nc.get("allow_tf32", True))
        torch.backends.cudnn.allow_tf32 = bool(nc.get("allow_tf32", True))
        torch.backends.cudnn.deterministic = bool(nc.get("deterministic", True))
        torch.backends.cudnn.benchmark = bool(nc.get("cudnn_benchmark", False))

# --- apply setup
set_seed(CONFIG["SEED"])
configure_torch_numeric(CONFIG["NUMERIC"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GEN = torch.Generator().manual_seed(CONFIG["SEED"])
DATALOADER_KW = {
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
    "worker_init_fn": seed_worker,
    "generator": GEN,
}

# Common normalization constants (bind transforms later per backbone policy)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Device: {DEVICE} | torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")


Device: cuda | torch 2.9.0+cu126 | CUDA: True


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
# ================================
# Phase 1 / Cell 2.0 — Architecture & Dataset Toggle
# ================================


# Architecture Toggle
# ================================
# Choose backbone: 'cnn' or 'vgg16_bn'
CONFIG.setdefault("BACKBONE", "cnn")  # default cnn
BACKBONE = str(CONFIG["BACKBONE"]).lower().strip()
assert BACKBONE in {"cnn", "vgg16_bn"}, f"Invalid BACKBONE={CONFIG['BACKBONE']}"
print(f"Backbone selected: {BACKBONE}")


# Dataset Toggle
# ================================
# Choose dataset: 'celeba' or 'cifar10'
CONFIG.setdefault("DATASET", "celeba")  # default : celeba
DATASET = str(CONFIG["DATASET"]).lower().strip()
assert DATASET in {"celeba", "cifar10"}, f"Invalid DATASET={CONFIG['DATASET']}"
print(f"Dataset selected: {DATASET}")

Backbone selected: cnn
Dataset selected: celeba


In [ ]:
# ================================
# Phase 1 / Cell 2A — 5-Layer CNN (architecture + init)
# ================================
import torch.nn as nn

class CNN_V1(nn.Module):
    def __init__(self, num_classes=CONFIG["NUM_CLASSES"], pools=5):
        super().__init__()
        self.pools = pools
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(inplace=True),  nn.MaxPool2d(2),
            nn.Conv2d(64,128,3, padding=1),  nn.BatchNorm2d(128), nn.ReLU(inplace=True),  nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),  nn.BatchNorm2d(256), nn.ReLU(inplace=True),  nn.MaxPool2d(2),
            nn.Conv2d(256,512,3,padding=1),  nn.BatchNorm2d(512), nn.ReLU(inplace=True),  nn.MaxPool2d(2),
            nn.Conv2d(512,512,3,padding=1),  nn.BatchNorm2d(512), nn.ReLU(inplace=True),  nn.MaxPool2d(2),
        )
        h, w = CONFIG["IMG_SIZE"]
        down = 2 ** self.pools
        flat_dim = (h // down) * (w // down) * 512
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(flat_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def _init_he(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)

if BACKBONE == "cnn":
    model = CNN_V1().to(DEVICE)
    model.apply(_init_he)
    # record the first Linear index in classifier (dynamic, not hard-coded)
    LINEAR1_INDEX = next(i for i, layer in enumerate(model.classifier) if isinstance(layer, nn.Linear))
    print(f"CNN_V1 ready — params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} — linear1@classifier[{LINEAR1_INDEX}]")


CNN_V1 ready — params: 29,606,914 — linear1@classifier[2]


In [ ]:
# ================================
# Phase 1 / Cell 2B — VGG16-BN (architecture + init)
# ================================

import torch.nn as nn
from torchvision.models import vgg16_bn, VGG16_BN_Weights

def build_vgg16_bn(num_classes: int, pretrained: bool = True) -> nn.Module:
    m = vgg16_bn(weights=VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None)
    in_feats = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_feats, num_classes)  # replace head
    if pretrained:
        # only init the replaced layer
        nn.init.kaiming_normal_(m.classifier[-1].weight, nonlinearity="relu")
        nn.init.zeros_(m.classifier[-1].bias)
    else:
        # training from scratch: He init conv/linear
        def _init_he(x):
            if isinstance(x, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(x.weight, nonlinearity="relu")
                if x.bias is not None:
                    nn.init.zeros_(x.bias)
        m.apply(_init_he)
    return m

if BACKBONE == "vgg16_bn":
    model = build_vgg16_bn(CONFIG["NUM_CLASSES"], pretrained=True).to(DEVICE)
    # record the first Linear index in classifier (don’t assume magic numbers)
    LINEAR1_INDEX = next(i for i, layer in enumerate(model.classifier) if isinstance(layer, nn.Linear))
    print(f"VGG16-BN ready — params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} — linear1@classifier[{LINEAR1_INDEX}]")

if BACKBONE == "vgg16_bn":
    CONFIG.setdefault("TAYLOR_MAX_BATCHES", 8)


In [ ]:
# ================================
# Phase 1 / Cell 3 — Dataset Download & Unpack
# ================================

import os, sys, shutil, json
from pathlib import Path

# ---- Config ----
CONFIG.setdefault("DATA_DIR", "./data")
DATA_ROOT = Path(CONFIG["DATA_DIR"]).resolve()

# ---- Shared Helper ----
def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

# ---- Define DATASET for this cell ----
DATASET = CONFIG.get("DATASET", "celeba").lower()


# Branch by DATASET (set in Cell 1.5)
# ================================

if DATASET == "celeba":

    # CelebA: Kaggle + Drive cache (UNCHANGED)
    # ================================================================
    CELEBA_DIR = DATA_ROOT / "celeba"
    KAGGLE_DS = "jessicali9530/celeba-dataset"
    PERSIST_TO_DRIVE = True

    def _find_image_dir(root: Path):
        for dp, _, files in os.walk(root):
            if sum(f.lower().endswith(".jpg") for f in files) > 1000:
                return dp
        return None

    def _find_celeba_root(base: Path):
        for dp, _, files in os.walk(base):
            fs = set(f.lower() for f in files)
            if "list_attr_celeba.csv" in fs and "list_eval_partition.csv" in fs:
                return Path(dp)
        return None

    def _kaggle_ready():
        # 1) Local ~/.kaggle
        if (Path.home()/".kaggle"/"kaggle.json").exists():
            return "local"
        # 2) Drive (only if already mounted)
        drive_kj = Path("/content/drive/MyDrive/kaggle/kaggle.json")
        if drive_kj.exists():
            os.environ["KAGGLE_CONFIG_DIR"] = str(drive_kj.parent)
            return "drive"
        return None

    def _persist_uploaded_to_drive(local_kj: Path):
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            drive_dir = Path("/content/drive/MyDrive/kaggle")
            _ensure_dir(drive_dir)
            shutil.copy2(local_kj, drive_dir / "kaggle.json")
            print("💾 Saved kaggle.json to Drive:/MyDrive/kaggle/kaggle.json for future runs.")
        except Exception as e:
            print(f"⚠️ Could not persist creds to Drive: {e}")

    # ---- Fast path: skip work if already present ----
    _ensure_dir(CELEBA_DIR)
    existing_root = _find_celeba_root(CELEBA_DIR)
    if existing_root and _find_image_dir(CELEBA_DIR):
        CONFIG["DATA_DIR"] = str(DATA_ROOT)   # keep consistent for downstream cells
        print(f"✅ CelebA ready at: {existing_root}")
    else:
        # ---- Ensure Kaggle creds with least friction ----
        mode = _kaggle_ready()
        if mode is None:
            try:
                # Upload once; then optionally persist to Drive.
                from google.colab import files
                print("➡️ Please upload your kaggle.json (Account → Create API Token)")
                uploaded = files.upload()
                if "kaggle.json" not in uploaded:
                    raise RuntimeError("kaggle.json not uploaded.")
                kj_local_dir = Path.home()/".kaggle"
                _ensure_dir(kj_local_dir)
                (kj_local_dir/"kaggle.json").write_bytes(uploaded["kaggle.json"])
                os.chmod(kj_local_dir/"kaggle.json", 0o600)
                print("✅ Kaggle creds installed locally.")
                if PERSIST_TO_DRIVE:
                    _persist_uploaded_to_drive(kj_local_dir/"kaggle.json")
            except Exception as e:
                raise RuntimeError(f"Failed to provision Kaggle credentials: {e}") from e
        else:
            print(f"✅ Using Kaggle creds from: {mode}")

        # ---- Download (idempotent) ----
        # Use --unzip to avoid separate unzip step; harmless if already downloaded.
        _ensure_dir(CELEBA_DIR)
        print("⬇️ Downloading CelebA (skips existing files)…")
        # Note: -q for quiet; remove -q if you want logs
        !kaggle datasets download -d {KAGGLE_DS} -p "{CELEBA_DIR}" --unzip -q

        # If Kaggle unzipped into a nested /celeba-dataset directory, flatten it
        nested = CELEBA_DIR / "celeba-dataset"
        if nested.exists() and nested.is_dir():
            for item in nested.iterdir():
                shutil.move(str(item), str(CELEBA_DIR))
            shutil.rmtree(nested, ignore_errors=True)

        # Final sanity: discover root again
        existing_root = _find_celeba_root(CELEBA_DIR)
        if not existing_root:
            raise RuntimeError("CelebA metadata not found after download. Check kaggle output.")
        if not _find_image_dir(CELEBA_DIR):
            raise RuntimeError("CelebA images folder not found after download.")

        CONFIG["DATA_DIR"] = str(DATA_ROOT)
        print(f"✅ CelebA ready at: {existing_root}")

elif DATASET == "cifar10":

    # CIFAR-10: Auto-download via torchvision
    # ================================================================
    from torchvision.datasets import CIFAR10

    _ensure_dir(DATA_ROOT)

    # Trigger download (torchvision caches automatically)
    print("⬇️ Downloading CIFAR-10 (if not cached)...")
    _ = CIFAR10(root=str(DATA_ROOT), train=True, download=True)
    _ = CIFAR10(root=str(DATA_ROOT), train=False, download=True)

    print(f"✅ CIFAR-10 ready at: {DATA_ROOT / 'cifar-10-batches-py'}")

else:
    raise ValueError(f"Unsupported DATASET='{DATASET}'. Must be 'celeba' or 'cifar10'.")

➡️ Please upload your kaggle.json (Account → Create API Token)


Saving kaggle.json to kaggle.json
✅ Kaggle creds installed locally.
Mounted at /content/drive
💾 Saved kaggle.json to Drive:/MyDrive/kaggle/kaggle.json for future runs.
⬇️ Downloading CelebA (skips existing files)…
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
✅ CelebA ready at: /content/data/celeba


In [ ]:
# ================================
# Phase 1 / Cell 4 — Data Pipeline (robust, stratified, decoupled)
# ================================

import os, random, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ---- Defaults ----
CONFIG.setdefault("DATA_DIR", "./data")
CONFIG.setdefault("IMG_SIZE", (224, 224))  # FIXED for apples-to-apples
CONFIG.setdefault("BATCH_SIZE", 64)
CONFIG.setdefault("TRAIN_SAMPLES", 10000)
CONFIG.setdefault("VAL_SAMPLES_EVAL", None)     # None => full validation
CONFIG.setdefault("PRUNE_REF_SAMPLES", 6000)

# Note: Seeding already done in Cell 1 with CONFIG["SEED"]
# No need to re-seed here - maintains single source of truth

# ---- Define DATASET for this cell ----
DATASET = CONFIG.get("DATASET", "celeba").lower()

# ---- Shared loader construction (used by both datasets) ----
def build_loaders(train_ds, eval_ds, prune_ref_ds, config):
    """
    Builds all required DataLoaders with consistent configuration.

    Args:
        train_ds: Training dataset
        eval_ds: Evaluation dataset
        prune_ref_ds: Pruning reference dataset
        config: CONFIG dict with BATCH_SIZE, SEED

    Returns:
        dict with keys: train_loader, eval_loader, prune_ref_loader,
                        bn_recal_loader, sequential_loader
    """
    PIN = bool(torch.cuda.is_available() or (
        hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    ))

    # Reuse generator and worker seeding from Cell 1 for consistency
    g = torch.Generator().manual_seed(config["SEED"])
    num_workers = 2
    pw_flag = num_workers > 0

    # Reuse seed_worker logic from Cell 1
    def _seed_worker(wid):
        s = config["SEED"] + wid
        random.seed(s)
        np.random.seed(s)

    train_loader = DataLoader(
        train_ds, batch_size=config["BATCH_SIZE"], shuffle=True,
        num_workers=num_workers, pin_memory=PIN,
        worker_init_fn=_seed_worker,
        generator=g, persistent_workers=pw_flag
    )
    eval_loader = DataLoader(
        eval_ds, batch_size=config["BATCH_SIZE"], shuffle=False,
        num_workers=num_workers, pin_memory=PIN,
        worker_init_fn=_seed_worker,
        generator=g, persistent_workers=pw_flag
    )
    prune_ref_loader = DataLoader(
        prune_ref_ds, batch_size=config["BATCH_SIZE"], shuffle=False,
        num_workers=num_workers, pin_memory=PIN,
        worker_init_fn=_seed_worker,
        generator=g, persistent_workers=pw_flag
    )

    bn_recal_loader = prune_ref_loader
    sequential_loader = DataLoader(
        eval_ds, batch_size=config["BATCH_SIZE"], shuffle=False,
        num_workers=0, pin_memory=PIN,
        worker_init_fn=_seed_worker,
        generator=g
    )

    return {
        "train_loader": train_loader,
        "eval_loader": eval_loader,
        "prune_ref_loader": prune_ref_loader,
        "bn_recal_loader": bn_recal_loader,
        "sequential_loader": sequential_loader,
    }


# Branch by DATASET (set in Cell 1.5)
# ================================

if DATASET == "celeba":

    # CelebA Pipeline (UNCHANGED)
    # ================================================================

    # ---- Locate CelebA contents (Cell 3 must have prepared these) ----
    root = os.path.join(CONFIG["DATA_DIR"], "celeba")
    img_dir = None
    for dp, _, files in os.walk(root):
        if sum(f.lower().endswith(".jpg") for f in files) > 1000:
            img_dir = dp
            break
    assert img_dir is not None, "CelebA images folder not found. Run Cell 3 first."

    attr_csv = os.path.join(root, "list_attr_celeba.csv")
    part_csv = os.path.join(root, "list_eval_partition.csv")
    assert os.path.exists(attr_csv) and os.path.exists(part_csv), \
        "CelebA metadata missing. Run Cell 3 to download/unpack."

    # ---- Read labels/partitions ----
    attrs = pd.read_csv(attr_csv)
    parts = pd.read_csv(part_csv)
    df = attrs.merge(parts, on="image_id")
    if "Male" not in df.columns:
        raise ValueError("'Male' attribute not found in list_attr_celeba.csv")

    # Convert Male {-1,+1} → {0,1}
    df["y"] = ((df["Male"] + 1) // 2).astype(int)
    df["partition"] = df["partition"].astype(int)

    train_pool = df[df["partition"] == 0].copy()
    valid_pool = df[df["partition"] == 1].copy()

    # ---- Helpers: stratified balanced sampling WITHOUT groupby.apply ----
    def stratified_balanced_sample_binary(frame: pd.DataFrame, n_total: int, seed: int) -> pd.DataFrame:
        """Aim for ~50/50 across y∈{0,1}; top-up from the other class if needed, then pad."""
        n_total = int(n_total)
        if n_total <= 0 or len(frame) == 0:
            return frame.iloc[[]].copy()

        half = n_total // 2
        cls0 = frame[frame["y"] == 0]
        cls1 = frame[frame["y"] == 1]

        rng0 = np.random.default_rng(seed)
        rng1 = np.random.default_rng(seed + 1)

        take0 = min(len(cls0), half)
        take1 = min(len(cls1), half)

        # top-up if one class underflows
        if take0 < half:
            deficit = half - take0
            take1 = min(len(cls1), half + deficit)
        elif take1 < half:
            deficit = half - take1
            take0 = min(len(cls0), half + deficit)

        parts = []
        if take0 > 0:
            parts.append(cls0.sample(n=take0, random_state=seed))
        if take1 > 0:
            parts.append(cls1.sample(n=take1, random_state=seed + 1))

        out = pd.concat(parts, axis=0) if parts else frame.iloc[[]]

        # final pad if still short
        if len(out) < n_total:
            remaining = frame.drop(out.index)
            need = n_total - len(out)
            if need > 0 and len(remaining) > 0:
                pad = remaining.sample(n=min(need, len(remaining)), random_state=seed + 2)
                out = pd.concat([out, pad], axis=0)

        return out.sample(frac=1.0, random_state=seed + 3).reset_index(drop=True)

    def print_split_stats(name: str, frame: pd.DataFrame):
        n = len(frame)
        c0 = int((frame["y"] == 0).sum()) if n else 0
        c1 = int((frame["y"] == 1).sum()) if n else 0
        p0 = 100.0 * c0 / max(1, n)
        p1 = 100.0 * c1 / max(1, n)
        print(f"  {name:<16}: {n:6d}  | 0={c0:5d} ({p0:4.1f}%)  1={c1:5d} ({p1:4.1f}%)")

    # ---- Build splits ----
    # (A) Train (balanced)
    n_train = min(CONFIG["TRAIN_SAMPLES"], len(train_pool))
    train_df_strat = stratified_balanced_sample_binary(train_pool, n_train, CONFIG["SEED"])

    # (B) Eval (full valid or capped + balanced)
    if CONFIG["VAL_SAMPLES_EVAL"] is None:
        eval_df_full = valid_pool.copy().reset_index(drop=True)
    else:
        cap = min(CONFIG["VAL_SAMPLES_EVAL"], len(valid_pool))
        eval_df_full = stratified_balanced_sample_binary(valid_pool, cap, CONFIG["SEED"] + 10)

    # (C) Prune-ref (balanced; prefer train, top-up from valid)
    want = CONFIG["PRUNE_REF_SAMPLES"]
    partA = stratified_balanced_sample_binary(train_pool, min(want, len(train_pool)), CONFIG["SEED"] + 20)
    if len(partA) < want:
        need = want - len(partA)
        partB = stratified_balanced_sample_binary(valid_pool, min(need, len(valid_pool)), CONFIG["SEED"] + 21)
        prune_ref_df = pd.concat([partA, partB], axis=0).sample(frac=1.0, random_state=CONFIG["SEED"] + 22).reset_index(drop=True)
    else:
        prune_ref_df = partA
    if len(prune_ref_df) > want:
        prune_ref_df = prune_ref_df.iloc[:want].reset_index(drop=True)

    # ---- Report
    print("✅ Splits created (sizes & class balance):")
    print_split_stats("train_df_strat", train_df_strat)
    print_split_stats("eval_df_full",  eval_df_full)
    print_split_stats("prune_ref_df",  prune_ref_df)

    # ---- Transforms (arch-agnostic; ImageNet stats) ----
    eval_transforms = transforms.Compose([
        transforms.Resize(CONFIG["IMG_SIZE"]),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    train_transforms = eval_transforms

    # ---- Dataset ----
    class CelebACustom(Dataset):
        def __init__(self, frame, img_dir, transform=None):
            self.df = frame.reset_index(drop=True)
            self.img_dir = img_dir
            self.transform = transform

        def __len__(self): return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            path = os.path.join(self.img_dir, row["image_id"])
            img = Image.open(path).convert("RGB")
            label = int(row["y"])
            if self.transform: img = self.transform(img)
            return img, label

    # ---- Datasets ----
    train_ds     = CelebACustom(train_df_strat, img_dir, train_transforms)
    eval_ds      = CelebACustom(eval_df_full,   img_dir, eval_transforms)
    prune_ref_ds = CelebACustom(prune_ref_df,   img_dir, eval_transforms)

    # ---- Build loaders via shared helper ----
    loaders = build_loaders(train_ds, eval_ds, prune_ref_ds, CONFIG)
    globals().update(loaders)

    # ---- Export DataFrames (for logging/debugging) ----
    globals().update({
        "train_df_strat": train_df_strat,
        "eval_df_full": eval_df_full,
        "prune_ref_df": prune_ref_df,
    })

    # ---- Smoke test ----
    imgs, labels = next(iter(train_loader))
    print(f"✅ Loaders ready. train_batch={tuple(imgs.shape)}, labels in {{0,1}}")
    print(f"  pin_memory={train_loader.pin_memory}, workers={train_loader.num_workers}")

elif DATASET == "cifar10":

    # CIFAR-10 Pipeline (NEW)
    # ================================================================
    from torchvision.datasets import CIFAR10

    print(f"--- CIFAR-10 Data Pipeline (IMG_SIZE={CONFIG['IMG_SIZE']}) ---")

    # ---- 1) Load datasets (already downloaded in Cell 3) ----
    base_train = CIFAR10(root=CONFIG["DATA_DIR"], train=True, download=False)
    base_test = CIFAR10(root=CONFIG["DATA_DIR"], train=False, download=False)

    # ---- 2) Auto-select or validate CIFAR_CLASSES ----
    if "CIFAR_CLASSES" not in CONFIG:
        # Auto-select top 2 classes by distribution
        targets_train = np.array(base_train.targets)
        counts = np.bincount(targets_train, minlength=10)
        top2 = np.argsort(counts)[-2:][::-1]  # descending order
        CONFIG["CIFAR_CLASSES"] = (int(top2[0]), int(top2[1]))
        print(f"   Auto-selected classes: {CONFIG['CIFAR_CLASSES']}")
        print(f"   Class {top2[0]}: {counts[top2[0]]:,} samples")
        print(f"   Class {top2[1]}: {counts[top2[1]]:,} samples")
    else:
        # Validate manual selection
        c0, c1 = CONFIG["CIFAR_CLASSES"]
        assert 0 <= c0 <= 9 and 0 <= c1 <= 9, \
            f"CIFAR_CLASSES must be in [0,9], got {(c0, c1)}"
        assert c0 != c1, \
            f"CIFAR_CLASSES must be distinct, got {(c0, c1)}"
        print(f"   Using manual class selection: {c0}, {c1}")

    c0, c1 = CONFIG["CIFAR_CLASSES"]

    # ---- 3) Filter to binary classes (optimized with NumPy masks) ----
    targets_train = np.array(base_train.targets)
    mask_train = (targets_train == c0) | (targets_train == c1)
    train_idx_all = np.where(mask_train)[0]
    train_labels = (targets_train[train_idx_all] == c1).astype(int)  # remap: c0→0, c1→1

    targets_test = np.array(base_test.targets)
    mask_test = (targets_test == c0) | (targets_test == c1)
    test_idx_all = np.where(mask_test)[0]
    test_labels = (targets_test[test_idx_all] == c1).astype(int)

    # ---- Validate filtered dataset is not empty ----
    if len(train_idx_all) == 0 or len(test_idx_all) == 0:
        raise ValueError(
            f"No samples found for CIFAR_CLASSES={CONFIG['CIFAR_CLASSES']}. "
            f"Train: {len(train_idx_all)}, Test: {len(test_idx_all)}"
        )

    print(f"  Binary filtering: {len(train_idx_all)} train, {len(test_idx_all)} test")

    # ---- 4) Stratified sampling helper (mirrors CelebA logic) ----
    def stratified_sample_indices(indices, labels, n_total, seed):
        """
        Stratified balanced sampling for indices (same logic as CelebA).
        Aims for 50/50 class balance, with top-up if one class underflows.
        """
        n_total = int(n_total)
        if n_total <= 0 or len(indices) == 0:
            return indices[:0]

        n_total = min(n_total, len(indices))
        rng = np.random.default_rng(seed)

        idx0 = indices[labels == 0]
        idx1 = indices[labels == 1]
        half = n_total // 2

        take0 = min(len(idx0), half)
        take1 = min(len(idx1), half)

        # Top-up if one class underflows
        if take0 < half:
            deficit = half - take0
            take1 = min(len(idx1), half + deficit)
        elif take1 < half:
            deficit = half - take1
            take0 = min(len(idx0), half + deficit)

        sel0 = rng.choice(idx0, take0, replace=False) if take0 > 0 else np.array([], dtype=idx0.dtype)
        sel1 = rng.choice(idx1, take1, replace=False) if take1 > 0 else np.array([], dtype=idx1.dtype)
        sampled = np.concatenate([sel0, sel1])
        rng.shuffle(sampled)
        return sampled

    # ---- 5) Build splits (same semantic roles as CelebA) ----
    n_train = min(CONFIG["TRAIN_SAMPLES"], len(train_idx_all))
    train_idx = stratified_sample_indices(train_idx_all, train_labels, n_train, CONFIG["SEED"])

    if CONFIG.get("VAL_SAMPLES_EVAL") is None:
        eval_idx = test_idx_all  # full test set
    else:
        cap = min(CONFIG["VAL_SAMPLES_EVAL"], len(test_idx_all))
        eval_idx = stratified_sample_indices(test_idx_all, test_labels, cap, CONFIG["SEED"] + 10)

    want_ref = CONFIG["PRUNE_REF_SAMPLES"]
    prune_idx = stratified_sample_indices(train_idx_all, train_labels, want_ref, CONFIG["SEED"] + 20)

    # ---- 6) Build full-length binary label arrays (for stats printing) ----
    train_binary_full = np.full(len(targets_train), -1, dtype=int)
    train_binary_full[train_idx_all] = train_labels

    test_binary_full = np.full(len(targets_test), -1, dtype=int)
    test_binary_full[test_idx_all] = test_labels

    # ---- 7) Print split stats (mirrors CelebA format) ----
    def print_split_stats_cifar(name, sampled_indices, full_binary_labels):
        """
        Print split statistics in same format as CelebA.

        Args:
            name: Split name (e.g., "train_sampled")
            sampled_indices: Indices into original dataset
            full_binary_labels: Binary labels for entire dataset (0/1 for binary, -1 for others)
        """
        n = len(sampled_indices)
        if n == 0:
            print(f"  {name:<16}: {n:6d}  | 0=    0 (  0.0%)  1=    0 (  0.0%)")
            return

        labels = full_binary_labels[sampled_indices]
        c0 = int((labels == 0).sum())
        c1 = int((labels == 1).sum())
        p0 = 100.0 * c0 / n
        p1 = 100.0 * c1 / n
        print(f"  {name:<16}: {n:6d}  | 0={c0:5d} ({p0:4.1f}%)  1={c1:5d} ({p1:4.1f}%)")

    print("✅ Splits created (sizes & class balance):")
    print_split_stats_cifar("train_sampled", train_idx, train_binary_full)
    print_split_stats_cifar("eval_sampled", eval_idx, test_binary_full)
    print_split_stats_cifar("prune_ref_sampled", prune_idx, train_binary_full)

    # ---- 8) Transforms (FIXED: 224x224 + ImageNet normalization) ----
    eval_transforms = transforms.Compose([
        transforms.Resize(CONFIG["IMG_SIZE"]),  # 32x32 → 224x224 upsampling
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    train_transforms = eval_transforms

    # ---- 9) Dataset wrapper (outputs {0,1} labels) ----
    class CifarBinary(Dataset):
        """
        Wraps CIFAR-10 for binary classification.
        Remaps class_a → 0, class_b → 1 on-the-fly.
        """
        def __init__(self, base, indices, class_a, class_b, transform=None):
            self.base = base
            self.indices = indices
            self.class_a = class_a
            self.class_b = class_b
            self.transform = transform

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, i):
            idx = int(self.indices[i])
            img, y = self.base[idx]
            label = 0 if y == self.class_a else 1
            if self.transform:
                img = self.transform(img)
            return img, label

    # ---- 10) Instantiate datasets ----
    train_ds = CifarBinary(base_train, train_idx, c0, c1, train_transforms)
    eval_ds = CifarBinary(base_test, eval_idx, c0, c1, eval_transforms)
    prune_ref_ds = CifarBinary(base_train, prune_idx, c0, c1, eval_transforms)

    # ---- 11) Build loaders via shared helper ----
    loaders = build_loaders(train_ds, eval_ds, prune_ref_ds, CONFIG)
    globals().update(loaders)

    # ---- 12) Smoke test ----
    imgs, labels = next(iter(train_loader))
    print(f"✅ Loaders ready. train_batch={tuple(imgs.shape)}, labels in {{0,1}}")
    print(f"  pin_memory={train_loader.pin_memory}, workers={train_loader.num_workers}")

else:
    raise ValueError(f"Unsupported DATASET='{DATASET}'. Must be 'celeba' or 'cifar10'.")

✅ Splits created (sizes & class balance):
  train_df_strat  :  10000  | 0= 5000 (50.0%)  1= 5000 (50.0%)
  eval_df_full    :  19867  | 0=11409 (57.4%)  1= 8458 (42.6%)
  prune_ref_df    :   6000  | 0= 3000 (50.0%)  1= 3000 (50.0%)
✅ Loaders ready. train_batch=(64, 3, 224, 224), labels in {0,1}
  pin_memory=True, workers=2


# Phase 2



In [ ]:
# ================================
# Phase 2 / Cell 1 — Compat checks & output paths (agnostic)
# ================================
import os, time, json, hashlib
import torch
import torch.nn as nn

# ---- Runtime flags (determinism/speed) ----
CONFIG.setdefault("DETERMINISTIC", True)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = CONFIG["DETERMINISTIC"]
    torch.backends.cudnn.benchmark     = not CONFIG["DETERMINISTIC"]

# ---- Model↔pipeline shape sanity (CNN only) ----
# VGG16_BN uses fixed 7x7 AAP -> 25088 in-features, so no rebuild needed.
if CONFIG.get("BACKBONE") == "cnn":
    h, w = CONFIG["IMG_SIZE"]
    with torch.no_grad():
        x = torch.zeros(1, 3, h, w, device=DEVICE)
        y = model.features(x)  # [1, C, H', W']
    C, Hs, Ws = int(y.shape[1]), int(y.shape[2]), int(y.shape[3])
    exp_flat = C * Hs * Ws

    try:
        cur_in = model.classifier[LINEAR1_INDEX].in_features  # set in Cell 2A/2B
    except Exception:
        cur_in = None

    if cur_in != exp_flat:
        # Silent, safe rebuild to match current IMG_SIZE / feature downscale
        model = CNN_V1(num_classes=CONFIG["NUM_CLASSES"]).to(DEVICE)
        # Recommended: refresh LINEAR1_INDEX for robustness (dynamic discovery policy)
        LINEAR1_INDEX = next(i for i, layer in enumerate(model.classifier) if isinstance(layer, nn.Linear))


# ---- Smoke test util (no auto-run) ----
criterion = nn.CrossEntropyLoss()

def _cuda_sync_if_needed(device: torch.device) -> bool:
    return torch.cuda.is_available() and isinstance(device, torch.device) and device.type == "cuda"

def run_smoke_test(
    model,
    loader=None,
    device=DEVICE,
    criterion: nn.Module | None = None,
    per_layer_timing: bool = False,
    do_backward: bool = True,
) -> bool:
    """Single-batch forward (and optional backward) to confirm compatibility."""
    if loader is None:
        loader = globals().get("train_loader") or globals().get("eval_loader")
    if loader is None:
        print("❌ Smoke Test: no loader available.")
        return False

    model.to(device).train()
    try:
        data, target = next(iter(loader))
    except Exception:
        print("❌ Smoke Test: loader empty or not iterable.")
        return False

    data   = data.to(device, non_blocking=True)
    target = target.to(device, non_blocking=True)

    do_sync = _cuda_sync_if_needed(device)
    if do_sync: torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(data)
    if do_sync: torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1e3

    # Basic shape & quick loss/backward
    if out.ndim != 2 or out.size(0) != target.size(0):
        print(f"❌ Smoke Test: output shape {tuple(out.shape)} mismatches batch {tuple(target.shape)}.")
        return False
    if criterion is not None and do_backward:
        model.zero_grad(set_to_none=True)
        loss = criterion(out, target)
        loss.backward()

    # Optional light per-layer timing
    if per_layer_timing and hasattr(model, "features"):
        with torch.no_grad():
            x = data
            for name, layer in model.features.named_children():
                if do_sync: torch.cuda.synchronize()
                t1 = time.perf_counter()
                x = layer(x)
                if do_sync: torch.cuda.synchronize()
                _ = (time.perf_counter() - t1) * 1e3  # keep silent to avoid spam

    # Minimal confirmation print
    preds = out.argmax(dim=1)
    acc   = (preds == target).float().mean().item()
    print(f"✅ Smoke: ok | fwd={t_ms:.1f} ms | batch={tuple(data.shape)} | acc≈{acc:.3f}")
    return True

# ---- Output paths (stable & collision-safe) ----
OUTPUT_ROOT = "./output"
PHASE       = "phase_2"
dataset_tag = DATASET  # Use the toggle from Cell 1.5
backbone    = CONFIG.get("BACKBONE", "cnn")

# Compact fingerprint of the run config
cfg_hash = hashlib.md5(json.dumps({
    "backbone": backbone,
    "img_size": CONFIG["IMG_SIZE"],
    "batch":    CONFIG["BATCH_SIZE"],
    "seed":     CONFIG["SEED"],
    "dataset":  dataset_tag,
}, sort_keys=True).encode()).hexdigest()[:8]

RUN_NAME = f"{PHASE}-{backbone}-{dataset_tag}-{cfg_hash}"
LOG_DIR  = os.path.join(OUTPUT_ROOT, RUN_NAME, "logs")
CKPT_DIR = os.path.join(OUTPUT_ROOT, RUN_NAME, "checkpoints")
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

CONFIG["RUN_NAME"]  = RUN_NAME
CONFIG["LOG_PATH"]  = os.path.join(LOG_DIR,  "log.csv")
CONFIG["SAVE_DIR"]  = CKPT_DIR

print(f"Paths ready → LOG:{CONFIG['LOG_PATH']} | CKPT:{CONFIG['SAVE_DIR']}")

Paths ready → LOG:./output/phase_2-cnn-celeba-a6cb478a/logs/log.csv | CKPT:./output/phase_2-cnn-celeba-a6cb478a/checkpoints


In [ ]:
#================================
# Phase 2 / Cell 2 — Train & Evaluate
#================================

from tqdm import tqdm
import os, time, json
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch import amp  # NEW: modern AMP API

# --------- Defaults & toggles (non-destructive) ---------
CONFIG.setdefault("EPOCHS", 5)
CONFIG.setdefault("LR", 3e-4)
CONFIG.setdefault("WEIGHT_DECAY", 1e-4)
CONFIG.setdefault("SCHEDULER", "cosine")          # {"none","cosine"}
CONFIG.setdefault("FREEZE_WARMUP_EPOCHS", 1 if str(CONFIG.get("BACKBONE","")).startswith("vgg") else 0)
CONFIG.setdefault("AMP", bool(torch.cuda.is_available()))
CONFIG.setdefault("GRAD_CLIP_NORM", None)         # e.g., 1.0 or None

assert "LOG_PATH" in CONFIG and "SAVE_DIR" in CONFIG, "Run Phase 2 / Cell 1 first (paths)."

def append_log(entry: dict):
    exists = os.path.exists(CONFIG["LOG_PATH"])
    pd.DataFrame([entry]).to_csv(CONFIG["LOG_PATH"], index=False, mode="a", header=not exists)

def maybe_compile(m):
    use = bool(CONFIG.get("USE_COMPILE", False))
    if use and hasattr(torch, "compile"):
        return torch.compile(m)
    return m

def _params(model, trainable_only=True):
    return (p for p in model.parameters() if (p.requires_grad or not trainable_only))

def _has_features(model):
    return hasattr(model, "features") and isinstance(model.features, nn.Module)

def _make_optimizer(lr: float):
    return optim.AdamW(_params(model, trainable_only=True), lr=lr, weight_decay=CONFIG["WEIGHT_DECAY"])

def _make_scheduler(optimizer):
    if CONFIG["SCHEDULER"] == "cosine" and CONFIG["EPOCHS"] > 0:
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["EPOCHS"])
    return None

# Optional quick smoke (if Cell 1 defined it)
if "run_smoke_test" in globals():
    _ = run_smoke_test(model, train_loader, DEVICE, criterion=nn.CrossEntropyLoss())

# --------- Prepare model/optimizer/AMP ---------
model = maybe_compile(model)
criterion = nn.CrossEntropyLoss()
freeze_warmup_epochs = int(CONFIG["FREEZE_WARMUP_EPOCHS"])

if freeze_warmup_epochs > 0 and _has_features(model):
    for p in model.features.parameters():
        p.requires_grad = False

optimizer = _make_optimizer(CONFIG["LR"])
scheduler = _make_scheduler(optimizer)

amp_enabled = bool(CONFIG["AMP"])
scaler = amp.GradScaler('cuda', enabled=amp_enabled)  # NEW: replaces torch.cuda.amp.GradScaler

best_val_acc, best_path = 0.0, None
os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    # Unfreeze once after warm-up
    if epoch == freeze_warmup_epochs + 1 and freeze_warmup_epochs > 0 and _has_features(model):
        for p in model.features.parameters():
            p.requires_grad = True
        optimizer = _make_optimizer(CONFIG["LR"] * 0.1)

    # ---- Train ----
    model.train()
    t0 = time.time()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for imgs, labels in tqdm(train_loader, desc=f"[Train] {epoch}/{CONFIG['EPOCHS']}"):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            with amp.autocast('cuda'):  # NEW: replaces torch.cuda.amp.autocast()
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            if CONFIG["GRAD_CLIP_NORM"] is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP_NORM"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            if CONFIG["GRAD_CLIP_NORM"] is not None:
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP_NORM"])
            optimizer.step()

        # FIX: avoid converting a grad-tracking tensor directly to float
        loss_val = loss.detach().item()
        train_loss += loss_val * imgs.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    if scheduler is not None:
        scheduler.step()

    train_loss /= max(1, train_total)
    train_acc   = train_correct / max(1, train_total)

    # ---- Validate ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for imgs, labels in tqdm(valid_loader if "valid_loader" in globals() else eval_loader,
                                 desc=f"[ Val ] {epoch}/{CONFIG['EPOCHS']}"):
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            if amp_enabled:
                with amp.autocast('cuda'):
                    logits = model(imgs)
                    loss = criterion(logits, labels)
            else:
                logits = model(imgs); loss = criterion(logits, labels)

            val_loss += loss.detach().item() * imgs.size(0)   # FIX here too
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= max(1, val_total)
    val_acc   = val_correct / max(1, val_total)

    append_log({
        "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
        "phase": "phase_2",
        "epoch": epoch,
        "batch_size": CONFIG["BATCH_SIZE"],
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_loss, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_acc":   val_acc,
        "epoch_time_s": time.time() - t0
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_path = os.path.join(CONFIG["SAVE_DIR"], f"best_epoch{epoch:02d}_acc{val_acc:.4f}.pth")
        torch.save(model.state_dict(), best_path)
        CONFIG["BASELINE_CKPT"] = best_path
        try:
            import shutil
            stable = os.path.join(CONFIG["SAVE_DIR"], "baseline_latest.pth")
            if os.path.abspath(stable) != os.path.abspath(best_path):
                shutil.copyfile(best_path, stable)
            with open(os.path.join(CONFIG["SAVE_DIR"], "baseline_info.json"), "w") as f:
                json.dump({"path": best_path, "epoch": epoch, "val_acc": float(val_acc),
                           "time": pd.Timestamp.now().isoformat()}, f, indent=2)
        except Exception:
            pass

print(f"Best Val Acc: {best_val_acc:.4f}")
if "BASELINE_CKPT" in CONFIG:
    print(f"Baseline ckpt: {CONFIG['BASELINE_CKPT']}")

✅ Smoke: ok | fwd=269.5 ms | batch=(64, 3, 224, 224) | acc≈0.453


[ Val ] 5/5: 100%|██████████| 311/311 [00:25<00:00, 12.17it/s]


Best Val Acc: 0.9416
Baseline ckpt: ./output/phase_2-cnn-celeba-a6cb478a/checkpoints/best_epoch05_acc0.9416.pth


# Phase 3

In [ ]:
# ================================
# Phase 3 / Cell 1 — Local Toggles (Parity+Production)
# ================================
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

# Safety: CONFIG/BACKBONE must already exist from Phase 1
try:
    BACKBONE = str(CONFIG.get("BACKBONE", "cnn")).lower()
except NameError:
    raise RuntimeError("Run Phase 1 first to define CONFIG/BACKBONE.")


def _default_alt_order() -> List[Dict[str, Any]]:
    """
    Pruning order per backbone, deep→shallow within blocks.
    Paths are dotted indices within model.features; bn=None where absent.
    """
    if BACKBONE == "vgg16_bn":
        # torchvision vgg16_bn indices:
        # conv @ [0,3], pool 6 | [7,10], pool 13 | [14,17,20], pool 23
        # [24,27,30], pool 33 | [34,37,40], pool 43
        return CONFIG.get("ALT_ORDER", [
            # Block 5 (pool=43)
            {"conv":"features.40","bn":"features.41","pool":"features.43","max_merges":None},
            {"conv":"features.37","bn":"features.38","pool":"features.43","max_merges":None},
            {"conv":"features.34","bn":"features.35","pool":"features.43","max_merges":None},
            # Block 4 (pool=33)
            {"conv":"features.30","bn":"features.31","pool":"features.33","max_merges":None},
            {"conv":"features.27","bn":"features.28","pool":"features.33","max_merges":None},
            {"conv":"features.24","bn":"features.25","pool":"features.33","max_merges":None},
            # Block 3 (pool=23)
            {"conv":"features.20","bn":"features.21","pool":"features.23","max_merges":None},
            {"conv":"features.17","bn":"features.18","pool":"features.23","max_merges":None},
            {"conv":"features.14","bn":"features.15","pool":"features.23","max_merges":None},
            # Block 2 (pool=13)
            {"conv":"features.10","bn":"features.11","pool":"features.13","max_merges":None},
            {"conv":"features.7", "bn":"features.8", "pool":"features.13","max_merges":None},
            # Block 1 (pool=6)
            {"conv":"features.3", "bn":"features.4", "pool":"features.6", "max_merges":None},
            {"conv":"features.0", "bn":"features.1", "pool":"features.6", "max_merges":None},
        ])
    else:
        # 5-layer CNN_V1 (conv,bn,pool triplets)
        return CONFIG.get("ALT_ORDER", [
            {"conv":"features.16","bn":"features.17","pool":"features.19","max_merges":None},  # B5
            {"conv":"features.12","bn":"features.13","pool":"features.15","max_merges":None},  # B4
            {"conv":"features.8", "bn":"features.9", "pool":"features.11","max_merges":None},  # B3
            {"conv":"features.4", "bn":"features.5", "pool":"features.7", "max_merges":None},  # B2
            {"conv":"features.0", "bn":"features.1", "pool":"features.3", "max_merges":None},  # B1
        ])

def _default_floors() -> Dict[str, int]:
    """
    Survivor floors per conv | Override via CONFIG['LAYER_SURVIVOR_FLOOR'] if needed.
    """
    if BACKBONE == "vgg16_bn":
        base = {
            # Block 5 (512C)
            "features.40": 256,
            "features.37": 256,
            "features.34": 256,
            # Block 4 (512C)
            "features.30": 256,
            "features.27": 256,
            "features.24": 256,
            # Block 3 (256C)
            "features.20": 192,
            "features.17": 192,
            "features.14": 192,
            # Block 2 (128C)
            "features.10": 112,
            "features.7" : 112,
            # Block 1 (64C)
            "features.3":  56,
            "features.0":  56,
        }
        return CONFIG.get("LAYER_SURVIVOR_FLOOR", base)
    else:
        base = {
            "features.16": 160,
            "features.12": 128,
            "features.8":  192,
            "features.4":  112,
            "features.0":  60,
        }
        return CONFIG.get("LAYER_SURVIVOR_FLOOR", base)


@dataclass
class LocalToggles:

    # ---- Core Configuration ----
    ALT_ORDER: List[Dict[str, Any]] = field(default_factory=_default_alt_order)
    LAYER_SURVIVOR_FLOOR: Dict[str, int] = field(default_factory=_default_floors)

    # ----- DATA / BATCH -----
    EVAL_BATCH_LIMIT:  int = int(CONFIG.get("EVAL_BATCH_LIMIT", 4000))
    BATCH_SIZE_EVAL:   int = int(CONFIG.get("BATCH_SIZE_EVAL", 512))

    B2_MULTI_LAYER:    bool = bool(CONFIG.get("B2_MULTI_LAYER", True))
    SIM_BATCH_LIMIT:   int = int(CONFIG.get("SIM_BATCH_LIMIT", 6000))
    PRUNE_REF_SAMPLES: int = int(CONFIG.get("PRUNE_REF_SAMPLES", 6000))  # for parity prints

    # ---- Active-set ----
    ACTIVE_SET_ENABLE:      bool  = bool(CONFIG.get("ACTIVE_SET_ENABLE", True))
    ACTIVE_MAX_SAMPLES:     int   = int(CONFIG.get("ACTIVE_MAX_SAMPLES", 3000))
    ACTIVE_MIN_SAMPLES:     int   = int(CONFIG.get("ACTIVE_MIN_SAMPLES", 1024))
    ACTIVE_REFRESH_EVERY:   int   = int(CONFIG.get("ACTIVE_REFRESH_EVERY", 150))

    ACTIVE_PER_CLASS:       int   = int(CONFIG.get("ACTIVE_PER_CLASS", 512))
    ACTIVE_STRICT_EQUAL:    bool  = bool(CONFIG.get("ACTIVE_STRICT_EQUAL", True))
    # Legacy (unused) knobs – kept for easy rollback:
    # ACTIVE_STRICT_PER_CLASS:int   = int(CONFIG.get("ACTIVE_STRICT_PER_CLASS", 512))
    # ACTIVE_MARGIN_TAU:      float = float(CONFIG.get("ACTIVE_MARGIN_TAU", 0.02))


    # ---- Cache / Loader policy (MOD.1) ----
    EVAL_CACHE_DEVICE: str = CONFIG.get("EVAL_CACHE_DEVICE", "device")  # {'cpu_pinned','device'}
    GAP_LOADER_SRC:    str = CONFIG.get("GAP_LOADER_SRC", "sequential")     # {'sequential','prune_ref'}
    EVAL_DTYPE:        str = CONFIG.get("EVAL_DTYPE", "float32")
    NON_BLOCKING_XFER: bool = bool(CONFIG.get("NON_BLOCKING_XFER", True))

    # ---- Similarity / linkage ----
    SIM_METRIC: str = CONFIG.get("SIM_METRIC", "pearson")    # {'cosine','pearson'}
    LINKAGE:    str = CONFIG.get("LINKAGE", "centroid")     # {'centroid','single','complete','average'}

    # ---- Gate / decision rule ----
    # BENEFIT_MODE/ZERO_AT remain from earlier variants but are
    # not used in the current local pruning core; keep commented
    # for quick rollback if needed.
    # BENEFIT_MODE:      str   = CONFIG.get("BENEFIT_MODE", "single_drop_cache")
    # ZERO_AT:           str   = CONFIG.get("ZERO_AT", "conv")  # {'conv','pool'}
    RATIO_MIN:         float = float(CONFIG.get("RATIO_MIN", 0.8))
    # MAGNITUDE_ENABLE/FIXED_M_* are legacy knobs not used in S5 core.
    # MAGNITUDE_ENABLE:  bool  = bool(CONFIG.get("MAGNITUDE_ENABLE", False))
    CALIB_PCTL:        int   = int(CONFIG.get("CALIB_PCTL", 60))
    CALIB_MIN_FLIPS:   int   = int(CONFIG.get("CALIB_MIN_FLIPS", 2))
    # FIXED_M_ENABLE:    bool  = bool(CONFIG.get("FIXED_M_ENABLE", False))
    # FIXED_M_MIN:       float = float(CONFIG.get("FIXED_M_MIN", 0.10))
    MAX_PAIRS_PROBED_PER_STEP: int = int(CONFIG.get("MAX_PAIRS_PROBED_PER_STEP", 128))

    # ---- Tier 2 Crossover (Feature Map GA + Regression) ----
    TIER2_ENABLE: bool        = bool(CONFIG.get("TIER2_ENABLE", True))
    TIER2_REGRESS_STEPS: int  = int(CONFIG.get("TIER2_REGRESS_STEPS", 50))
    TIER2_REGRESS_LR: float   = float(CONFIG.get("TIER2_REGRESS_LR", 0.01))

    # ----- WILSON CI / early-exit evidence guards -----
    Z_WILSON:          float = float(CONFIG.get("Z_WILSON", 2.0))
    MIN_USED_ACCEPT:   int   = int(CONFIG.get("MIN_USED_ACCEPT", 512))
    MIN_USED_REJECT:   int   = int(CONFIG.get("MIN_USED_REJECT", 1024))
    MIN_FLIPS_FRAC:    float = float(CONFIG.get("MIN_FLIPS_FRAC", 0.002))

    # ---- Intrinsic Consistency & drift guards ----
    # IC_* knobs are from the older IC z-stop guard and
    # are no longer used in S5; keep commented for rollback.
    # IC_ENABLE:           bool  = bool(CONFIG.get("IC_ENABLE", True))
    # IC_Z_MAX:            float = float(CONFIG.get("IC_Z_MAX", 4.0))
    # IC_WARMUP_ACCEPTS:   int   = int(CONFIG.get("IC_WARMUP_ACCEPTS", 40))
    # IC_MIN_DIST_SAMPLES: int   = int(CONFIG.get("IC_MIN_DIST_SAMPLES", 30))

    PROFILE_HC:         bool  = bool(CONFIG.get("PROFILE_HC", True))
    PROBE_M_SAMPLES:    int   = int(CONFIG.get("PROBE_M_SAMPLES", 64))
    # Legacy drift guards (not used in S5 core):
    # MID_EVAL_DRIFT_THRESH: float = float(CONFIG.get("MID_EVAL_DRIFT_THRESH", 0.01))
    MID_EVAL_SAMPLE:       int   = int(CONFIG.get("MID_EVAL_SAMPLE", 3000))
    # PERCLASS_DRIFT_THRESH: float = float(CONFIG.get("PERCLASS_DRIFT_THRESH", 0.03))
    # PERCLASS_CONSEC_N:     int   = int(CONFIG.get("PERCLASS_CONSEC_N", 2))

    # ---- Tail heuristics & rollback ----
    TAIL_LAST_FRAC:         float = float(CONFIG.get("TAIL_LAST_FRAC", 0.15))
    RATIO_MIN_TAIL_DELTA:   float = float(CONFIG.get("RATIO_MIN_TAIL_DELTA", 0.05))
    # IC_Z_* and ROLLBACK_* are legacy from S1 and not used by
    # S5’s safety net; keep commented for easy rollback.
    # IC_Z_TAIL_DELTA:        float = float(CONFIG.get("IC_Z_TAIL_DELTA", 0.4))
    # IC_Z_MIN_FLOOR:         float = float(CONFIG.get("IC_Z_MIN_FLOOR", 1.0))
    # ROLLBACK_MAX_UNDO:      int   = int(CONFIG.get("ROLLBACK_MAX_UNDO", 20))
    # ROLLBACK_ACC_DROP:      float = float(CONFIG.get("ROLLBACK_ACC_DROP", 0.1))
    # ROLLBACK_PERCLASS_DROP: float = float(CONFIG.get("ROLLBACK_PERCLASS_DROP", 0.03))

    # ---- Global rails (vs. baseline, layer-agnostic) ----
    GLOBAL_MAX_OVERALL_DROP: float = float(CONFIG.get("GLOBAL_MAX_OVERALL_DROP", 0.02))
    GLOBAL_MAX_CLASS_DROP:   float = float(CONFIG.get("GLOBAL_MAX_CLASS_DROP",   0.06))
    MAX_LAYER_DROP: float = float(CONFIG.get("MAX_LAYER_DROP", 0.01))
    # ---- Safety Brake (Preventive) ----
    SAFETY_BRAKE_ENABLE: bool = True
    SAFETY_BRAKE_INTERVAL: int = 20  # Check every 20 merges
    SAFETY_BRAKE_ACC_DROP: float = 0.02 # Stop if drop exceeds 2% (Global limit)

    # ---- BN cadence (guarded by HAS_BN in core) ----
    BN_CALIB_TOTAL:          int   = int(CONFIG.get("BN_CALIB_TOTAL", 2048))
    BN_MOMENTUM:             float = float(CONFIG.get("BN_MOMENTUM", 0.05))
    BN_MINI_ITERS:           int   = int(CONFIG.get("BN_MINI_ITERS", 2048))
    BN_FULL_ITERS:           int   = int(CONFIG.get("BN_FULL_ITERS", 2048))
    STARTUP_MINI_BN_ENABLE:  bool  = bool(CONFIG.get("STARTUP_MINI_BN_ENABLE", False))
    BN_REFRESH_EVERY_MERGES: int   = int(CONFIG.get("BN_REFRESH_EVERY_MERGES", 200))
    BN_REFRESH_EVERY_LAYER:  bool  = bool(CONFIG.get("BN_REFRESH_EVERY_LAYER", True))
    BN_MAX_MINI_PER_LAYER:   int   = int(CONFIG.get("BN_MAX_MINI_PER_LAYER", 3))
    BN_ENFORCE_MIN_CADENCE:  bool  = bool(CONFIG.get("BN_ENFORCE_MIN_CADENCE", True))
    BN_MIN_CADENCE:          int   = int(CONFIG.get("BN_MIN_CADENCE", 200))
    ALLOW_SMALL_CADENCE:     bool  = bool(CONFIG.get("ALLOW_SMALL_CADENCE", False))
    _BN_CADENCE_CLAMPED:     bool  = field(default=False, init=False)



    # ---- Housekeeping / validation ----
    VALIDATE_PATHS: bool = bool(CONFIG.get("VALIDATE_PATHS", True))
    HEAP_PARTIAL_REFRESH_TOPK: int = int(CONFIG.get("HEAP_PARTIAL_REFRESH_TOPK", 5000))
    # IC_REPROBE_PAIRS: int = int(CONFIG.get("IC_REPROBE_PAIRS", 48))
    MIN_SIM_STOP: float = float(CONFIG.get("MIN_SIM_STOP", 0.70))
    # VERBOSE: bool = bool(CONFIG.get("VERBOSE", True))
    ENABLE_COMPILE: bool = bool(CONFIG.get("ENABLE_COMPILE", True))

    # ---- Proximity surface (hybrid, MOD.1) ----
    # PROX_ALLPAIRS_THRESHOLD: int = int(CONFIG.get("PROX_ALLPAIRS_THRESHOLD", 128))
    PROX_MAX_INIT_PAIRS: Optional[int] = CONFIG.get("PROX_MAX_INIT_PAIRS", None)  # cap for large C

    # ==== MOD.1 policy flags & diagnostics ====
    # The following MOD.1 flags are not used by S5’s local core;
    # keep commented for potential future use.
    # COMPILE_TIMING:   str  = CONFIG.get("COMPILE_TIMING", "post_gates")  # {'post_gates','early'}
    # RECOMPILE_FINAL:  bool = bool(CONFIG.get("RECOMPILE_FINAL", True))
    # LINEAR1_DETECT:   str  = CONFIG.get("LINEAR1_DETECT", "dynamic")     # {'dynamic','fixed'}
    # LINEAR1_SLICE_MODE: str = CONFIG.get("LINEAR1_SLICE_MODE", "dynamic_from_features")
    # NEXT_CONV_RESOLVER: str = CONFIG.get("NEXT_CONV_RESOLVER", "conv_relative")  # {'conv_relative','pool_anchored'}
    COMPILED_EVAL:    bool = bool(CONFIG.get("COMPILED_EVAL", False))
    # MAX_MERGES: Optional[int] = CONFIG.get("MAX_MERGES", None)
    HC_HEARTBEAT_EVERY: int = int(CONFIG.get("HC_HEARTBEAT_EVERY", 20))   # 0 disables
    HC_HEARTBEAT_SECS: float = float(CONFIG.get("HC_HEARTBEAT_SECS", 30.0))

    def __post_init__(self):
        # BN cadence min
        if self.BN_ENFORCE_MIN_CADENCE and not self.ALLOW_SMALL_CADENCE:
            if self.BN_REFRESH_EVERY_MERGES < self.BN_MIN_CADENCE:
                object.__setattr__(self, "BN_REFRESH_EVERY_MERGES", self.BN_MIN_CADENCE)
                object.__setattr__(self, "_BN_CADENCE_CLAMPED", True)

        # Normalize EVAL_CACHE_DEVICE
        val = str(self.EVAL_CACHE_DEVICE).lower().strip()
        object.__setattr__(self, "EVAL_CACHE_DEVICE",
                           "cpu_pinned" if val in {"cpu", "pinned", "cpu-pinned", "cpu_pinned"} else "device")

        # Normalize/alias GAP loader mode
        src = str(self.GAP_LOADER_SRC).lower().strip()
        mode_cfg = str(CONFIG.get("GAP_LOADER_MODE", "")).lower().strip()
        if mode_cfg in {"sequential", "ref_if_available"}:
            normalized_src = "sequential" if mode_cfg == "sequential" else "prune_ref"
            object.__setattr__(self, "GAP_LOADER_SRC", normalized_src)
            object.__setattr__(self, "GAP_LOADER_MODE", mode_cfg)
        else:
            mode = "sequential" if src == "sequential" else "ref_if_available"
            object.__setattr__(self, "GAP_LOADER_MODE", mode)

        # Robust None parsing
        if isinstance(self.PROX_MAX_INIT_PAIRS, str) and self.PROX_MAX_INIT_PAIRS.strip().lower() == "none":
            object.__setattr__(self, "PROX_MAX_INIT_PAIRS", None)

# ARCH-SPECIFIC toggles (documented; harmless if unused)
LocalToggles.USE_VGG_BN     = bool(CONFIG.get("USE_VGG_BN", BACKBONE == "vgg16_bn"))
LocalToggles.FUSE_AT_DEPLOY = bool(CONFIG.get("FUSE_AT_DEPLOY", True))

# Keep expected symbol names for downstream cells
T = LocalToggles()


# Minimal local-approach defaults (merged; do not overwrite CONFIG-backed fields)
_LOCAL_PATCH = {
    "APPROACH": "local",
    "PROX_MAX_INIT_PAIRS": None,     # parity: no cap unless user sets one
    "HEAP_PARTIAL_REFRESH_TOPK": 10000,
    # Legacy knobs (unused in S5 core) – commented for easy rollback:
    # "MERGE_OP_EQUIV": "avg",
    # "MERGE_OP_NON_EQUIV": "drop",
    # "STARTUP_MINI_BN_ENABLE": False,
}
for _k, _v in _LOCAL_PATCH.items():
    if not hasattr(T, _k):
        setattr(T, _k, _v)

# Alias compatibility
if not hasattr(T, "EVAL_CACHE_POLICY"):
    setattr(T, "EVAL_CACHE_POLICY", T.EVAL_CACHE_DEVICE)
if not hasattr(T, "GAP_LOADER_MODE"):
    setattr(T, "GAP_LOADER_MODE", "sequential" if T.GAP_LOADER_SRC == "sequential" else "ref_if_available")

# ---- Aliases from MOD.1/PRD (keep backward-compat) ----
if "EVAL_CACHE_POLICY" in CONFIG and "EVAL_CACHE_DEVICE" not in CONFIG:
    CONFIG["EVAL_CACHE_DEVICE"] = CONFIG["EVAL_CACHE_POLICY"]

if "GAP_LOADER_MODE" in CONFIG and "GAP_LOADER_SRC" not in CONFIG:
    mode = str(CONFIG["GAP_LOADER_MODE"]).lower().strip()
    CONFIG["GAP_LOADER_SRC"] = "sequential" if mode == "sequential" else "prune_ref"

In [ ]:
# ================================
# Phase 3 / Cell 2 — Adapter, validators, loader pickers
# ================================
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple
import torch, torch.nn as nn

# -- dotted-path helpers (safe across ChannelGate insertions)
def get_module(model: nn.Module, dotted: str) -> nn.Module:
    cur = model
    for tok in dotted.split("."):
        cur = cur[int(tok)] if tok.isdigit() else getattr(cur, tok)
    return cur

def _get_parent_and_name(model: nn.Module, dotted: str) -> Tuple[nn.Module, str]:
    parts = dotted.split(".")
    parent = model
    for tok in parts[:-1]:
        parent = parent[int(tok)] if tok.isdigit() else getattr(parent, tok)
    return parent, parts[-1]

def get_conv_module(model: nn.Module, conv_path: str) -> nn.Conv2d:
    m = get_module(model, conv_path)
    if isinstance(m, nn.Sequential):
        assert isinstance(m[0], nn.Conv2d), f"Expected Conv2d at {conv_path}[0], got {type(m[0])}"
        return m[0]
    assert isinstance(m, nn.Conv2d), f"Expected Conv2d at {conv_path}, got {type(m)}"
    return m

def try_get_bn_module(model: nn.Module, bn_path: Optional[str]) -> Optional[nn.Module]:
    if not bn_path:
        return None
    try:
        m = get_module(model, bn_path)
        return m if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)) else None
    except Exception:
        return None

# -- adapter
@dataclass
class ModelAdapter:
    model: nn.Module
    device: torch.device
    T: Any

    # dynamic linear1 discovery (first Linear in classifier)
    def resolve_linear1(self) -> Tuple[nn.Linear, int]:
        assert hasattr(self.model, "classifier"), "expected `model.classifier` to exist"
        for i, layer in enumerate(self.model.classifier):
            if isinstance(layer, nn.Linear):
                return layer, i
        raise RuntimeError("No nn.Linear found in `model.classifier`")

    def has_bn(self) -> bool:
        return any(isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)) for m in self.model.modules())

    # robust "next conv" resolver: scan forward from the conv's parent container
    def next_conv_path_from_conv(self, conv_path: str) -> Optional[str]:
        parent_path = ".".join(conv_path.split(".")[:-1])
        leaf = conv_path.split(".")[-1]
        parent = get_module(self.model, parent_path)
        try:
            start_idx = int(leaf)
        except ValueError:
            return None

        def _is_conv_container(m: nn.Module):
            return isinstance(m, nn.Conv2d) or (isinstance(m, nn.Sequential) and len(m)>0 and isinstance(m[0], nn.Conv2d))

        for j in range(start_idx + 1, len(parent)):
            if _is_conv_container(parent[j]):
                return f"{parent_path}.{j}"
        return None  # terminal → linear1 slice

    # semantic validators for ALT_ORDER and floors
    def validate_paths_and_floors(self) -> None:
        ALT_ORDER = getattr(self.T, "ALT_ORDER", None)
        assert isinstance(ALT_ORDER, list) and ALT_ORDER, "T.ALT_ORDER must be a non-empty list."
        floors: Dict[str, int] = getattr(self.T, "LAYER_SURVIVOR_FLOOR", {})
        for item in ALT_ORDER:
            conv, pool = item["conv"], item["pool"]
            bn = item.get("bn", None)
            try:
                c = get_conv_module(self.model, conv)
            except Exception as e:
                raise AssertionError(f"[validate] bad conv path: {conv} ({e})")
            try:
                p = get_module(self.model, pool)
            except Exception as e:
                raise AssertionError(f"[validate] bad pool path: {pool} ({e})")
            if not hasattr(p, "forward"):
                raise AssertionError(f"[validate] pool is not a module: {pool}")
            if bn:
                bnm = try_get_bn_module(self.model, bn)
                if bnm is None:
                    raise AssertionError(f"[validate] bn path given but not BatchNorm: {bn}")
            floor = int(floors.get(conv, 0))
            if floor < 0 or floor > c.out_channels:
                raise AssertionError(f"[validate] floor {floor} invalid for {conv} with C={c.out_channels}")

    # loader routing: gap/eval sources via toggles
    def pick_loaders(self) -> Dict[str, Any]:
        gap_src = getattr(self.T, "GAP_LOADER_SRC", "sequential").lower()
        eval_loader = globals().get("sequential_loader") or globals().get("eval_loader")
        assert eval_loader is not None, "sequential/eval loader not found (from Phase-2)."
        prune_ref = globals().get("prune_ref_loader", None)
        if gap_src == "prune_ref" and prune_ref is not None:
            gap_loader = prune_ref
        else:
            gap_loader = eval_loader
        return {"eval": eval_loader, "gap": gap_loader}

# -- build adapter and validate
adapter = ModelAdapter(model=model, device=DEVICE, T=T)
if bool(getattr(T, "VALIDATE_PATHS", True)):
    adapter.validate_paths_and_floors()
LOADERS = adapter.pick_loaders()
HAS_BN = adapter.has_bn()
print(f"[Adapter] BN={HAS_BN} | GAP from: {'prune_ref_loader' if LOADERS['gap'] is globals().get('prune_ref_loader') else 'sequential_loader'}")


[Adapter] BN=True | GAP from: sequential_loader


In [ ]:
# ================================
# Phase 3 / Cell 3 — Shared primitives (Tier 2 Crossover + Pearson)
# ================================
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple, Any, Set
import time, heapq, math, random, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm import tqdm

# === Read-only local shorthands ===
BN_REFRESH_EVERY_MERGES   = int(getattr(T, "BN_REFRESH_EVERY_MERGES", 200))
BN_MAX_MINI_PER_LAYER     = int(getattr(T, "BN_MAX_MINI_PER_LAYER", 1))
BN_MINI_ITERS             = int(getattr(T, "BN_MINI_ITERS", 2048))
BN_FULL_ITERS             = int(getattr(T, "BN_FULL_ITERS", 2048))
MID_EVAL_SAMPLE           = int(getattr(T, "MID_EVAL_SAMPLE", 3000))
MID_EVAL_DRIFT_THRESH     = float(getattr(T, "MID_EVAL_DRIFT_THRESH", 0.01))
PERCLASS_DRIFT_THRESH     = float(getattr(T, "PERCLASS_DRIFT_THRESH", 0.03))
PERCLASS_CONSEC_N         = int(getattr(T, "PERCLASS_CONSEC_N", 2))
TAIL_LAST_FRAC            = float(getattr(T, "TAIL_LAST_FRAC", 0.15))
RATIO_MIN_TAIL_DELTA      = float(getattr(T, "RATIO_MIN_TAIL_DELTA", 0.05))
HEAP_PARTIAL_REFRESH_TOPK = int(getattr(T, "HEAP_PARTIAL_REFRESH_TOPK", 5000))
ACTIVE_REFRESH_EVERY      = int(getattr(T, "ACTIVE_REFRESH_EVERY", BN_REFRESH_EVERY_MERGES))
CALIB_PCTL                = int(getattr(T, "CALIB_PCTL", 60))
CALIB_MIN_FLIPS           = int(getattr(T, "CALIB_MIN_FLIPS", 2))
MIN_SIM_STOP              = float(getattr(T, "MIN_SIM_STOP", 0.0))
RATIO_MIN_BASE            = float(getattr(T, "RATIO_MIN", 0.80))
ROLLBACK_MAX_UNDO         = int(getattr(T, "ROLLBACK_MAX_UNDO", 20))

# --- INCREMENTAL ROLLBACK SETTINGS ---
INCREMENTAL_ROLLBACK      = True
ROLLBACK_STEPS            = 6
ROLLBACK_RATIO            = 0.15

# --- SAFETY BRAKE SETTINGS ---
SAFETY_BRAKE_ENABLE       = True
SAFETY_BRAKE_INTERVAL     = 25
SAFETY_BRAKE_ACC_DROP     = 0.02

# -------- Pretty print helpers --------
def _bar(title: str, ch: str = "═", width: int = 70):
    line = ch * width
    print(f"\n{line}\n{title}\n{line}")

def _rule(ch: str = "─", width: int = 70): print(ch * width)
def _fmt_pct(x: float, digits: int = 2):  return f"{x*100:.{digits}f}%"
def _fmt_pp(delta: float, digits: int = 2, sign=True):
    if delta is None: return "—"
    s = f"{delta*100:.{digits}f} pp"
    return f"{'+' if (sign and delta>=0) else ''}{s}" if sign else s
def _fmt_int(x: int): return f"{x:,}"
def _fmt_big(x: int): return f"{x:,}"

# -------- Stopwatch & profiler --------
class Stopwatch:
    def __init__(self): self.t = {}
    def go(self, key): self.t.setdefault(key, 0.0); self._s = time.perf_counter(); self._k = key
    def stop(self): self.t[self._k] += (time.perf_counter()-self._s)

class HCProfiler:
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.counters = {"heap_pops":0, "heap_pushes":0, "pairs_sim_recomputed":0,
                         "benefit_evals":0, "benefit_eval_s":0.0, "sim_update_s":0.0,
                         "tier2_attempts":0, "tier2_success":0, "tier2_regress_s":0.0}
    def inc(self, k, v=1):  self.enabled and self.counters.__setitem__(k, self.counters.get(k,0)+v)
    def add_time(self, k, dt): self.enabled and self.counters.__setitem__(k, self.counters.get(k,0.0)+dt)

prof = HCProfiler(getattr(T, "PROFILE_HC", False))
sw   = Stopwatch()

def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

# -------- ChannelGate (+ helpers) --------
class ChannelGate(nn.Module):
    def __init__(self, C: int):
        super().__init__()
        self.register_buffer("mask", torch.ones(1, C, 1, 1))
    def forward(self, x): return x * self.mask

def _ensure_gate_for_conv_path(model, conv_path: str, device) -> str:
    parent, name = _get_parent_and_name(model, conv_path)
    L = getattr(parent, name)
    if isinstance(L, nn.Sequential):
        for idx, child in enumerate(L):
            if isinstance(child, ChannelGate):
                if isinstance(L[0], nn.Conv2d) and child.mask.shape[1] != L[0].out_channels:
                    with torch.no_grad():
                        child.mask = torch.ones(1, L[0].out_channels, 1, 1, device=device)
                return f"{conv_path}.{idx}"
        conv = L[0]; gate = ChannelGate(conv.out_channels).to(device)
        setattr(parent, name, nn.Sequential(*list(L), gate))
        return f"{conv_path}.{len(L)}"
    else:
        assert isinstance(L, nn.Conv2d), f"Expected Conv2d at {conv_path}, got {type(L)}"
        gate = ChannelGate(L.out_channels).to(L.weight.device)
        setattr(parent, name, nn.Sequential(L, gate))
        return f"{conv_path}.1"

def _get_gate_by_conv_path(model, conv_path: str) -> ChannelGate:
    m = get_module(model, conv_path)
    if isinstance(m, nn.Sequential):
        for child in m:
            if isinstance(child, ChannelGate): return child
    if isinstance(m, nn.Conv2d):
        parent, name = _get_parent_and_name(model, conv_path)
        gate = ChannelGate(m.out_channels).to(m.weight.device)
        setattr(parent, name, nn.Sequential(m, gate))
        return gate
    raise RuntimeError(f"No ChannelGate at {conv_path}")

@torch.no_grad()
def set_drop_mask(model, conv_path: str, indices: Optional[torch.LongTensor]):
    gate = _get_gate_by_conv_path(model, conv_path)
    gate.mask.fill_(1.0)
    if indices is not None and indices.numel() > 0:
        gate.mask.index_fill_(1, indices.to(gate.mask.device), 0.0)

def strip_channel_gate_for_infer(model, conv_path: str) -> bool:
    parent, name = _get_parent_and_name(model, conv_path)
    L = getattr(parent, name)
    if isinstance(L, nn.Sequential):
        keep = [m for m in L if not isinstance(m, ChannelGate)]
        if len(keep) != len(L):
            setattr(parent, name, nn.Sequential(*keep))
            return True
    return False

# -------- Shapes & MACs --------
def count_params_module(m: nn.Module): return sum(p.numel() for p in m.parameters(recurse=True))
def conv_macs_per_image(conv: nn.Conv2d, C_in, H_out, W_out):
    kH,kW = conv.kernel_size if isinstance(conv.kernel_size, tuple) else (conv.kernel_size, conv.kernel_size)
    groups = conv.groups
    return conv.out_channels * H_out * W_out * (C_in//groups) * kH * kW
def linear_macs_per_image(l: nn.Linear): return l.in_features * l.out_features

def layer_shapes_prepool(model, device, conv_path, pool_path, loader):
    shapes = {}
    conv = get_conv_module(model, conv_path)
    pool = get_module(model, pool_path)
    def h_conv(_, __, out): shapes.setdefault("conv", out.shape); return out
    def h_pool(_, __, out): shapes.setdefault("pool", out.shape); return out
    h1 = conv.register_forward_hook(h_conv)
    h2 = pool.register_forward_hook(h_pool)
    try:
        probe_loader = torch.utils.data.DataLoader(loader.dataset,
                                                  batch_size=loader.batch_size,
                                                  shuffle=False, num_workers=0, pin_memory=False)
        imgs, _ = next(iter(probe_loader))
        model.eval()
        with torch.no_grad():
            if hasattr(model, "features"): _ = model.features(imgs.to(device, non_blocking=True))
            else: _ = model(imgs.to(device, non_blocking=True))
    finally:
        h1.remove(); h2.remove()
    return shapes

# -------- Eval cache --------
@torch.no_grad()
def build_eval_cache(loader, device, max_images=None):
    policy = getattr(T, "EVAL_CACHE_DEVICE", "cpu_pinned").lower()
    xs, ys, seen = [], [], 0
    pbar = tqdm(total=(max_images or len(loader.dataset)), desc="[Build Eval Cache]")
    for imgs, y in loader:
        take = imgs.size(0) if max_images is None else min(imgs.size(0), max_images - seen)
        if take <= 0: break
        if policy == "device":
            xs.append(imgs[:take].to(device, non_blocking=True))
        else:
            xs.append(imgs[:take].pin_memory() if hasattr(imgs, "pin_memory") else imgs[:take])
        ys.append(y[:take].clone())
        seen += take; pbar.update(take)
        if max_images and seen >= max_images: break
    pbar.close()
    X = torch.cat(xs, 0); y = torch.cat(ys, 0)
    return X, y

# -------- BN recalibration --------
@torch.no_grad()
def build_bn_calib_cache_from_eval(X_eval, y_eval, total=2048):
    classes = torch.unique(y_eval).tolist()
    per = max(1, total // max(1, len(classes)))
    idxs=[]
    for c in classes:
        cand = (y_eval==c).nonzero(as_tuple=False).squeeze(1)
        take = min(per, cand.numel()); idxs.append(cand[:take])
    return X_eval[torch.cat(idxs, 0)], y_eval[torch.cat(idxs, 0)]

@torch.no_grad()
def bn_recalibrate_from_cache(model, X_cache, iters=2048, device=DEVICE):
    if X_cache is None: return
    was_training = model.training
    model.train()
    original_momentum = []
    mom = float(getattr(T, "BN_MOMENTUM", 0.05))
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            original_momentum.append((m, m.momentum)); m.momentum = mom
    seen = 0; bs = int(getattr(T, "BATCH_SIZE_EVAL", 512))
    while seen < min(iters, X_cache.size(0)):
        xb = X_cache[seen:seen+bs].to(device, non_blocking=True)
        _ = model(xb)
        seen += xb.size(0)
    for m, old in original_momentum: m.momentum = old
    model.eval()
    if was_training: model.train()

# -------- Eval helpers --------
@torch.no_grad()
def eval_top1_and_perclass(model, loader, device):
    model.eval()
    ys, yh = [], []
    pbar = tqdm(total=len(loader.dataset), desc="[Eval Top1]")
    for x,y in loader:
        x = x.to(device, non_blocking=True)
        out = model(x)
        yh.append(out.argmax(1).detach().cpu()); ys.append(y.cpu())
        pbar.update(x.size(0))
    pbar.close()
    y = torch.cat(ys); yh = torch.cat(yh)
    acc = (y==yh).float().mean().item()
    classes = torch.unique(y).tolist(); per=[]
    for c in classes:
        m = (y==c)
        per.append((c, (yh[m]==c).float().mean().item() if m.any() else float('nan'), int(m.sum())))
    return acc, sorted(per, key=lambda t: t[0])

@torch.no_grad()
def quick_acc_on_cache(model, X_eval, y_eval, device, max_n, bs):
    n = min(int(max_n), X_eval.size(0))
    correct = 0
    for i in range(0, n, bs):
        xb = X_eval[i:i+bs].to(device, non_blocking=True)
        pred = model(xb).argmax(1).cpu()
        correct += int((pred == y_eval[i:i+bs]).sum())
    return correct / max(1, n)

@torch.no_grad()
def quick_perclass_on_cache(model, X_eval, y_eval, device, max_n, bs):
    n = min(int(max_n), X_eval.size(0))
    yh = []
    for i in range(0, n, bs):
        yh.append(model(X_eval[i:i+bs].to(device, non_blocking=True)).argmax(1).cpu())
    yh = torch.cat(yh); y = y_eval[:n].cpu()
    classes = torch.unique(y).tolist(); out = {}
    for c in classes:
        m = (y==c)
        if m.any(): out[c] = float((yh[m]==c).float().mean().item())
    return out

# -------- Active-set selection --------
@torch.no_grad()
def pick_active_indices_strict_equal(model, X, y, device, per_class: int, bs: int):
    classes = torch.unique(y).tolist(); logits = []
    for i in range(0, X.size(0), bs):
        logits.append(model(X[i:i+bs].to(device, non_blocking=True)).detach().cpu())
    L = torch.cat(logits, 0)
    top2 = torch.topk(L, k=2, dim=1).values
    margin = (top2[:,0] - top2[:,1]).abs()
    buckets = []
    for c in classes:
        idx_c = (y==c).nonzero(as_tuple=False).squeeze(1)
        order = torch.argsort(margin[idx_c])
        take = min(per_class, idx_c.numel())
        buckets.append(idx_c[order[:take]])
    return torch.cat(buckets, 0)

@torch.no_grad()
def pick_active_indices_stratified(model, X, y, device, max_total: int, per_class: int, bs: int, strict_equal: bool):
    if strict_equal:
        per = min(per_class, max_total // max(1, len(torch.unique(y))))
        return pick_active_indices_strict_equal(model, X, y, device, per, bs)
    logits = []
    for i in range(0, X.size(0), bs):
        logits.append(model(X[i:i+bs].to(device, non_blocking=True)).detach().cpu())
    L = torch.cat(logits, 0)
    top2 = torch.topk(L, k=2, dim=1).values
    margin = (top2[:,0] - top2[:,1]).abs()
    classes = torch.unique(y).tolist()
    if per_class * len(classes) > max_total: per_class = max_total // max(1, len(classes))
    buckets = []
    for c in classes:
        idx_c = (y==c).nonzero(as_tuple=False).squeeze(1)
        order = torch.argsort(margin[idx_c])
        buckets.append(idx_c[order[:per_class]])
    return torch.cat(buckets, 0)

# -------- GAP collection --------
@torch.no_grad()
def collect_gap_matrix(model, loader, device, pool_path, max_images=None):
    pool = get_module(model, pool_path)
    feats = []
    def hook(_, __, out): feats.append(out.mean(dim=(2,3)).detach().cpu()); return out
    h = pool.register_forward_hook(hook)
    model.eval(); seen = 0
    pbar = tqdm(total=(max_images or len(loader.dataset)), desc="[Collect GAP]")
    for imgs, _ in loader:
        imgs = imgs.to(device, non_blocking=True); _ = model.features(imgs) if hasattr(model, "features") else model(imgs)
        seen += imgs.size(0); pbar.update(imgs.size(0))
        if max_images and seen >= max_images: break
    pbar.close(); h.remove()
    return torch.cat(feats, dim=0)

# -------- Proximity (Pearson Centering) --------
def _normalize_cols(X: torch.Tensor) -> torch.Tensor: return F.normalize(X.float(), p=2, dim=0)
def _center_cols(X: torch.Tensor) -> torch.Tensor:    return X - X.mean(dim=0, keepdim=True)

class HCProximity:
    def __init__(self, X_gap: torch.Tensor, metric: str = "cosine", linkage: str = "centroid", max_init_pairs: Optional[int] = None):
        self.metric = metric.lower(); self.linkage = linkage.lower()
        # Enforcing Pearson by centering cols if metric is 'pearson'
        Xp = _center_cols(X_gap) if self.metric=="pearson" else X_gap
        self.base = Xp; self.N, self.C = Xp.shape
        self.members: Dict[int, List[int]] = {c: [c] for c in range(self.C)}
        self.vecs: Dict[int, torch.Tensor] = {c: _normalize_cols(Xp[:, c:c+1]).squeeze(1) for c in range(self.C)}
        self.live: Set[int] = set(range(self.C))
        self.heap: List[Tuple[float,int,int]] = []
        self.S: Dict[Tuple[int,int], float] = {}
        self._build_heap(max_init_pairs)

    def _pair_sim(self, i: int, j: int) -> float:
        if self.linkage == "centroid": return float((self.vecs[i]*self.vecs[j]).sum().item())
        sims=[]
        for a in self.members[i]:
            va = _normalize_cols(self.base[:, a:a+1]).squeeze(1)
            for b in self.members[j]:
                vb = _normalize_cols(self.base[:, b:b+1]).squeeze(1)
                sims.append(float((va*vb).sum().item()))
        if self.linkage == "single":   return max(sims)
        if self.linkage == "complete": return min(sims)
        return sum(sims)/len(sims)

    def _build_heap(self, max_pairs: Optional[int]):
        idx = sorted(self.live)
        total_pairs = (len(idx)*(len(idx)-1))//2
        cap = total_pairs if (max_pairs is None) else min(max_pairs, total_pairs)
        pbar = tqdm(total=cap, desc="[Proximity Build]")
        cnt = 0
        for a in range(len(idx)):
            i = idx[a]
            for b in range(a+1, len(idx)):
                if cnt >= cap: break
                j = idx[b]
                t0 = time.perf_counter(); sim = self._pair_sim(i, j); prof.add_time("sim_update_s", time.perf_counter()-t0)
                self.S[(i,j)] = sim
                heapq.heappush(self.heap, (-sim, i, j)); prof.inc("heap_pushes"); cnt += 1; pbar.update(1)
            if cnt >= cap: break
        pbar.close()

    def pop_best(self) -> Optional[Tuple[int,int,float]]:
        t0 = time.perf_counter()
        while self.heap:
            neg, i, j = heapq.heappop(self.heap); prof.inc("heap_pops")
            sim = -neg
            cur = self.S.get((min(i,j), max(i,j)), None)
            if (i in self.live) and (j in self.live) and (cur is not None) and (abs(cur - sim) < 1e-6):
                prof.add_time("sim_update_s", time.perf_counter()-t0)
                return i, j, sim
        prof.add_time("sim_update_s", time.perf_counter()-t0)
        return None

    def update_after_merge(self, keep: int, drop: int, op: str = "avg"):
        assert keep in self.live and drop in self.live
        self.live.remove(drop)
        if op == "avg":
            self.members[keep].extend(self.members[drop])
            cols = [self.base[:, c:c+1] for c in self.members[keep]]
            v = torch.mean(torch.cat(cols, 1), dim=1)
            self.vecs[keep] = F.normalize(v, p=2, dim=0)
        else:
            self.members[keep].extend(self.members[drop])
        for (a,b) in list(self.S.keys()):
            if (a==drop) or (b==drop) or (a==keep) or (b==keep):
                if (a,b) in self.S: del self.S[(a,b)]
        for k in list(self.live):
            if k == keep: continue
            i,j = (k, keep) if k < keep else (keep, k)
            t0 = time.perf_counter(); sim = self._pair_sim(i, j); prof.add_time("sim_update_s", time.perf_counter()-t0)
            self.S[(i,j)] = sim
            heapq.heappush(self.heap, (-sim, i, j)); prof.inc("heap_pushes")

    def partial_refresh(self, topk: int = 10000) -> int:
        if topk <= 0 or not self.heap: return 0
        cand, seen = [], set()
        for neg, i, j in self.heap:
            a, b = (i, j) if i < j else (j, i)
            if (a, b) in seen: continue
            if (i in self.live) and (j in self.live):
                cand.append((i, j)); seen.add((a, b))
                if len(cand) >= topk: break
        refreshed = 0
        for (i, j) in cand:
            ii, jj = (i, j) if i < j else (j, i)
            t0 = time.perf_counter(); sim = self._pair_sim(ii, jj); prof.add_time("sim_update_s", time.perf_counter() - t0)
            self.S[(ii, jj)] = sim; heapq.heappush(self.heap, (-sim, ii, jj)); prof.inc("heap_pushes")
            refreshed += 1
        return refreshed

# -------- Convenience: print proximity stats --------
def _print_distance_stats_from_heap(prox: "HCProximity", sample_k: int = 10000):
    vals = []
    for (i, j), sim in prox.S.items():
        vals.append(1.0 - float(sim))
        if len(vals) >= sample_k: break
    if not vals: return
    arr = np.array(vals, dtype=np.float64)
    mean = float(arr.mean()); std = float(arr.std()); med = float(np.median(arr))
    p90 = float(np.percentile(arr, 90)); p99 = float(np.percentile(arr, 99))
    top_sim = -prox.heap[0][0] if prox.heap else None
    print("[prox] distance stats (1 - sim) over sampled pairs:")
    print(f"  mean={mean:.4f}  std={std:.4f}  median={med:.4f}  p90={p90:.4f}  p99={p99:.4f}")
    if top_sim is not None: print(f"  current top-sim={top_sim:.4f} (top-dist={1.0-top_sim:.4f})")

# -------- Early-benefit test (Wilson CI) --------
Z_WILSON        = float(getattr(T, "Z_WILSON", 2.0))
MIN_USED_ACCEPT = int(getattr(T, "MIN_USED_ACCEPT", 512))
MIN_USED_REJECT = int(getattr(T, "MIN_USED_REJECT", 1024))
MIN_FLIPS_FRAC  = float(getattr(T, "MIN_FLIPS_FRAC", 0.002))

def _wilson_ci(phat: float, n: int, z: float = 2.0):
    if n <= 0: return 0.0, 1.0
    denom = 1.0 + (z*z)/n
    center = (phat + (z*z)/(2*n)) / denom
    margin = (z/denom) * math.sqrt(max(0.0, phat*(1.0-phat)/n + (z*z)/(4*n*n)))
    return center - margin, center + margin

@torch.no_grad()
def calibrate_m_floor(model, conv_path: str, S_all: set, X_eval: torch.Tensor, y_eval: torch.Tensor, device, batch_size: int, ratio_min: float) -> float:
    rng = np.random.default_rng(0)
    idx = sorted(S_all)
    max_pairs = min(int(getattr(T, "PROBE_M_SAMPLES", 64)), max(1, len(idx) * (len(idx) - 1) // 2))
    seen = set(); ms = []
    Xe = X_eval[:min(256, X_eval.size(0))]; Ye = y_eval[:min(256, y_eval.size(0))]
    while len(ms) < max_pairs and len(idx) >= 2:
        a, b = rng.choice(idx, size=2, replace=False)
        i, j = (a, b) if a < b else (b, a)
        key = (i, j)
        if key in seen: continue
        seen.add(key)
        n01, n10, used = benefit_pair_early(
            model, conv_path, S_all, i, j, Xe, Ye, device, batch_size=batch_size,
            ratio_min=ratio_min, min_abs_flips_override=None
        )
        ms.append((n01 + n10) / max(1, used))
    if not ms: return CALIB_MIN_FLIPS / max(1, X_eval.size(0))
    m_hat = float(np.percentile(ms, CALIB_PCTL))
    return max(m_hat, CALIB_MIN_FLIPS / max(1, X_eval.size(0)))

# === TIER 2 HELPER: Feature Map Crossover Generation ===
def generate_hybrid_fm(fm1: torch.Tensor, fm2: torch.Tensor) -> torch.Tensor:
    """
    Sorts pixels by activation magnitude, splits into X(High)/Y(Mid)/Z(Low) zones,
    and synthesizes a target 'Best of Both' feature map.
    Input: [B, H, W] (single channel slice per parent)
    Output: [B, H, W]
    """
    B, H, W = fm1.shape
    flat1 = fm1.view(B, -1)
    flat2 = fm2.view(B, -1)
    N = flat1.size(1)

    # Thresholds for zones (Top 25%, Mid 50%, Low 25%)
    top_k = int(0.25 * N)
    low_k = int(0.25 * N)

    # Sort by magnitude (descending)
    mag1 = flat1.abs()
    mag2 = flat2.abs()

    # Determine dominant parent for High-Energy Zone (X)
    # Heuristic: Whichever has higher total energy in its top-k pixels
    val1, _ = torch.topk(mag1, k=top_k, dim=1)
    val2, _ = torch.topk(mag2, k=top_k, dim=1)
    score1 = val1.sum(dim=1)
    score2 = val2.sum(dim=1)

    # Create masks (1 where parent 1 wins, 0 where parent 2 wins)
    p1_wins = (score1 >= score2).float().unsqueeze(1) # [B, 1]

    # Hybrid construction
    # Zone X (High): Take from dominant parent
    # Zone Y (Mid):  Take MAX(abs(p1), abs(p2)) (Union of features)
    # Zone Z (Low):  Take from recessive parent (Noise filling) - or just leave as is.
    # Simplified Crossover for Efficiency:
    # Target = Max(abs(p1), abs(p2)) * sign(dominant)
    # This effectively captures the "union" of active regions.
    target_mag = torch.maximum(mag1, mag2)

    # Restore sign? Regressing to magnitude is easier, but we need sign for correct convolution.
    # We take the sign of the parent with the larger magnitude at that pixel.
    mask_p1 = (mag1 >= mag2).float()
    target = (flat1 * mask_p1) + (flat2 * (1.0 - mask_p1))

    return target.view(B, H, W)

# === TIER 2 HELPER: Filter Regression ===
def regress_filter_weights(model, conv_path: str, target_fm: torch.Tensor,
                           i: int, j: int, X_input: torch.Tensor) -> torch.Tensor:
    """
    Solves for F_new such that Conv(Input, F_new) ~ Target_FM.
    Uses gradient descent optimization.
    """
    conv = get_conv_module(model, conv_path)
    kH, kW = conv.kernel_size
    Cin = conv.in_channels
    device = X_input.device

    # --- CRITICAL FIX: Detach target so we don't backprop through it ---
    target_fm = target_fm.detach()
    # -----------------------------------------------------------------

    # Initialize F_new as average of parents (Warm Start)
    w_i = conv.weight.data[i].detach()
    w_j = conv.weight.data[j].detach()
    w_init = 0.5 * (w_i + w_j)

    # Optimization target
    F_new = w_init.clone().requires_grad_(True)
    optimizer = torch.optim.Adam([F_new], lr=getattr(T, "TIER2_REGRESS_LR", 0.01))

    # Helper to run just this single filter convolution
    # We use F.conv2d directly to avoid overhead
    padding = conv.padding
    stride = conv.stride
    dilation = conv.dilation
    groups = 1 # Single filter is always group=1 relative to its input slice? No, input is full depth.

    steps = getattr(T, "TIER2_REGRESS_STEPS", 50)
    loss_fn = nn.MSELoss()

    # Pre-slice input if groups > 1 (Depthwise case handling if needed, assuming standard conv here)
    if conv.groups > 1:
        # Complex case skipped for simplicity; VGG/CNN uses groups=1 usually.
        # Fallback to average if groups > 1
        return w_init

    for _ in range(steps):
        # Forward pass: Conv(X, F_new)
        # weight shape must be [1, Cin, kH, kW]
        out = F.conv2d(X_input, F_new.unsqueeze(0), bias=None,
                       stride=stride, padding=padding, dilation=dilation)

        # Target FM is [B, H, W], Output is [B, 1, H, W]
        loss = loss_fn(out.squeeze(1), target_fm)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return F_new.detach()

# === DROP-IN: benefit_pair_early (Optimized) ===
@torch.no_grad()
def benefit_pair_early(model, conv_path: str, keep_set: set,
                       i: int, j: int,
                       X_eval: torch.Tensor, y_eval: torch.Tensor,
                       device, batch_size: int, ratio_min: float, min_abs_flips_override: Optional[int] = None) -> Tuple[int,int,int]:
    global prof
    t_all0 = time.perf_counter()
    prof.inc("benefit_evals", 1)

    C = get_conv_module(model, conv_path).out_channels
    dropped_base = sorted(set(range(C)) - set(keep_set))
    dropped_vec = torch.as_tensor(dropped_base, dtype=torch.long, device=device)

    gate = _get_gate_by_conv_path(model, conv_path)
    idx_i = torch.tensor([int(i)], device=device)
    idx_j = torch.tensor([int(j)], device=device)

    n01 = 0; n10 = 0; used = 0
    min_abs_flips = (int(min_abs_flips_override) if (min_abs_flips_override is not None)
                     else max(2, int(MIN_FLIPS_FRAC * X_eval.size(0))))

    base_mask_1 = torch.ones(1, C, 1, 1, device=device)
    if dropped_vec.numel() > 0:
        base_mask_1.index_fill_(1, dropped_vec, 0.0)

    orig_mask = gate.mask
    step_size = max(1, batch_size // 2)

    try:
        for s in range(0, X_eval.size(0), step_size):
            xb = X_eval[s:s+step_size].to(device, non_blocking=True).float()
            yb = y_eval[s:s+step_size].to(device, non_blocking=True)
            curr_bs = xb.size(0)

            xb2 = torch.cat([xb, xb], dim=0) # [2*curr_bs, C_in, H, W]
            mask2 = base_mask_1.repeat(2*curr_bs, 1, 1, 1)

            mask2[:curr_bs].index_fill_(1, idx_i, 0.0)
            mask2[curr_bs:].index_fill_(1, idx_j, 0.0)

            gate.mask = mask2
            logits2 = model(xb2)

            preds2 = logits2.argmax(dim=1)
            pi = preds2[:curr_bs]; pj = preds2[curr_bs:]

            Ai = (pi == yb); Aj = (pj == yb)
            n01 += int((~Ai &  Aj).sum().item())
            n10 += int(( Ai & ~Aj).sum().item())
            used += int(curr_bs)

            if (n01 + n10) > 0:
                bi = n10 / max(1, used); bj = n01 / max(1, used)
                big, sml = (bi, bj) if bi >= bj else (bj, bi)
                rhat = (sml / big) if big > 0 else 0.0
                lo, hi = _wilson_ci(rhat, used, z=Z_WILSON)

                if (lo >= ratio_min) and (used >= MIN_USED_ACCEPT) and ((n01+n10) >= min_abs_flips):
                    break
                if (hi <  ratio_min) and (used >= MIN_USED_REJECT):
                    break
            else:
                if used >= MIN_USED_ACCEPT:
                    break
    finally:
        gate.mask = orig_mask

    prof.add_time("benefit_eval_s", time.perf_counter() - t_all0)
    return n01, n10, used

def compute_r_m(n01, n10, N):
    bi = n10 / max(1,N); bj = n01 / max(1,N)
    big, sml = (bi,bj) if bi>=bj else (bj,bi)
    r = (sml/(big+1e-12)) if big>0 else 0.0
    m = (n01+n10)/max(1,N)
    return r, m, bi, bj, big, sml

def gate_decision_ratio_only(n01, n10, N, ratio_min: float) -> Tuple[bool, Dict[str,float], str]:
    if (n01 + n10) == 0:
        return True, {"r":1.0, "m":0.0, "bi":0.0, "bj":0.0}, "identical"
    r, m, bi, bj, _, _ = compute_r_m(n01, n10, N)
    ok = (r >= float(ratio_min))
    return ok, {"r":r,"m":m,"bi":bi,"bj":bj}, ("ratio_only" if ok else "reject")

# -------- Surgery (Optimized: Supports Tier 2 Custom Weights) --------
def prune_conv_block_and_linear_(model, device, conv_path, bn_path, pool_path,
                                 keep_indices: Set[int],
                                 merges_history: List[dict],
                                 eval_loader, adapter: ModelAdapter):
    lin1, lin1_idx = adapter.resolve_linear1()
    conv_seq_or_conv = get_module(model, conv_path)
    conv = conv_seq_or_conv[0] if isinstance(conv_seq_or_conv, nn.Sequential) else conv_seq_or_conv
    bn   = try_get_bn_module(model, bn_path)

    # Capture Old Channel Count (C_old) at start
    C_old = conv.out_channels

    keep = torch.as_tensor(sorted(list(keep_indices)), dtype=torch.long)

    if keep.numel() == conv.out_channels:
        print(f"[surgery] no-op: keep={keep.numel()} equals current out_channels"); return model

    with torch.no_grad():
        for m in merges_history:
            k, d = m["keep"], m["drop"]
            # TIER 2 CHECK: Do we have a regressed weight?
            if "custom_weight" in m and m["custom_weight"] is not None:
                # Apply custom weight to 'keep' index
                conv.weight.data[k] = m["custom_weight"].to(conv.weight.device)
                # Bias? Average bias is usually fine or regressed bias could be added.
                if conv.bias is not None:
                    conv.bias.data[k] = (conv.bias.data[k] + conv.bias.data[d]) * 0.5
            else:
                # TIER 3 (Drop/Standard Merge): Averaging
                conv.weight.data[k] = (conv.weight.data[k] + conv.weight.data[d]) * 0.5
                if conv.bias is not None:
                    conv.bias.data[k] = (conv.bias.data[k] + conv.bias.data[d]) * 0.5

            # BN updates (always average for stability)
            if bn is not None:
                bn.weight.data[k]       = (bn.weight.data[k] + bn.weight.data[d]) * 0.5
                bn.bias.data[k]         = (bn.bias.data[k] + bn.bias.data[d]) * 0.5
                bn.running_mean.data[k] = (bn.running_mean.data[k] + bn.running_mean.data[d]) * 0.5
                bn.running_var.data[k]  = (bn.running_var.data[k] + bn.running_var.data[d]) * 0.5

        conv.weight = nn.Parameter(conv.weight.data[keep].clone())
        if conv.bias is not None: conv.bias = nn.Parameter(conv.bias.data[keep].clone())
        conv.out_channels = keep.numel()
        if bn is not None:
            bn.weight = nn.Parameter(bn.weight.data[keep].clone())
            bn.bias   = nn.Parameter(bn.bias.data[keep].clone())
            bn.running_mean = bn.running_mean.data[keep].clone()
            bn.running_var  = bn.running_var.data[keep].clone()
            bn.num_features = keep.numel()

    if isinstance(conv_seq_or_conv, nn.Sequential) and len(conv_seq_or_conv)>=2 and isinstance(conv_seq_or_conv[1], ChannelGate):
        with torch.no_grad():
            conv_seq_or_conv[1].mask = torch.ones(1, conv.out_channels, 1, 1, device=device)

    next_conv_path = adapter.next_conv_path_from_conv(conv_path)
    if next_conv_path is not None:
        next_seq_or_conv = get_module(model, next_conv_path)
        next_conv = next_seq_or_conv[0] if isinstance(next_seq_or_conv, nn.Sequential) else next_seq_or_conv
        assert isinstance(next_conv, nn.Conv2d) and next_conv.groups == 1, "[surgery] Only standard Conv2d (groups=1) is supported."
        with torch.no_grad():
            W = next_conv.weight.data
            new_W = W[:, keep, :, :].clone()
            next_conv.weight = nn.Parameter(new_W)
            next_conv.in_channels = keep.numel()
        print(f"[surgery] internal: kept={keep.numel()} at {conv_path}; sliced fan-in of {next_conv_path} to Cin={keep.numel()}")
        return model

    shape={}
    pool = get_module(model, pool_path)
    def h_pool(_, __, out): shape.setdefault("shape", out.shape); return out
    h = pool.register_forward_hook(h_pool)
    with torch.no_grad():
        imgs,_ = next(iter(eval_loader)); _ = model.features(imgs.to(device)) if hasattr(model, "features") else model(imgs.to(device))
    h.remove()
    _, C_after, H, W = shape["shape"]; HW=H*W

    bb = globals().get("BACKBONE", "cnn").lower()
    if bb == "vgg16_bn":
        HW_real = lin1.in_features // C_old
        if HW_real != HW: HW = HW_real

    keep_cols=[]
    for c in keep.tolist(): keep_cols.extend(range(c*HW, (c+1)*HW))
    keep_cols = torch.as_tensor(keep_cols, dtype=torch.long)
    with torch.no_grad():
        new_lin1 = nn.Linear(keep_cols.numel(), lin1.out_features, bias=True).to(device)
        new_lin1.weight.copy_(lin1.weight.data[:, keep_cols]); new_lin1.bias.copy_(lin1.bias.data)
    model.classifier[lin1_idx] = new_lin1
    print(f"[surgery] terminal: kept={keep.numel()} channels; lin1.in_features={new_lin1.in_features}")
    return model

# --- NEW HELPER: Restore Structure from Snapshot ---
def _restore_model_structure(model, snapshot_state, layer_specs, device):
    """Restores conv/BN dimensions to match snapshot so load_state_dict succeeds."""
    for spec in layer_specs:
        conv_path = spec["conv"]
        bn_path = spec.get("bn")
        conv_seq_or_conv = get_module(model, conv_path)
        conv = conv_seq_or_conv[0] if isinstance(conv_seq_or_conv, nn.Sequential) else conv_seq_or_conv

        conv_key = f"{conv_path}.weight"
        if isinstance(conv_seq_or_conv, nn.Sequential): conv_key = f"{conv_path}.0.weight"

        if conv_key in snapshot_state:
            w = snapshot_state[conv_key]
            Cout, Cin = w.shape[0], w.shape[1]
            if conv.out_channels != Cout or conv.in_channels != Cin:
                new_conv = nn.Conv2d(Cin, Cout, conv.kernel_size, conv.stride, conv.padding, bias=(conv.bias is not None)).to(device)
                if isinstance(conv_seq_or_conv, nn.Sequential):
                    gate = conv_seq_or_conv[1] if len(conv_seq_or_conv)>1 else None
                    if gate:
                         gate.mask = torch.ones(1, Cout, 1, 1, device=device)
                         parent, name = _get_parent_and_name(model, conv_path)
                         setattr(parent, name, nn.Sequential(new_conv, gate))
                else:
                    parent, name = _get_parent_and_name(model, conv_path)
                    setattr(parent, name, new_conv)

        if bn_path:
            bn = get_module(model, bn_path)
            bn_key = f"{bn_path}.weight"
            if bn_key in snapshot_state:
                w = snapshot_state[bn_key]
                if bn.num_features != w.shape[0]:
                    bn.num_features = w.shape[0]
                    bn.weight = nn.Parameter(torch.empty_like(w))
                    bn.bias = nn.Parameter(torch.empty_like(w))
                    bn.running_mean = torch.empty_like(w)
                    bn.running_var = torch.empty_like(w)

    if "LINEAR1_INDEX" in globals():
        lin_idx = globals()["LINEAR1_INDEX"]
        lin1 = model.classifier[lin_idx]
        lin_key = None
        for k in snapshot_state:
            if k.endswith(f"classifier.{lin_idx}.weight"):
                lin_key = k; break
        if lin_key:
            w = snapshot_state[lin_key]
            din = w.shape[1]
            if lin1.in_features != din:
                new_lin = nn.Linear(din, lin1.out_features).to(device)
                model.classifier[lin_idx] = new_lin

# -------- Main per-layer runner (Updated: 3-TIER Logic) --------
def run_local_layer(model, pass_idx: int, conv_path: str, bn_path: Optional[str], pool_path: str,
                    X_eval, y_eval, X_bn_fixed, active_idx,
                    prof: HCProfiler, sw: Stopwatch, device, loaders: Dict[str,Any],
                    adapter: ModelAdapter, verbose=True, max_merges: Optional[int] = None,
                    global_baseline_acc: Optional[float] = None, global_baseline_per: Optional[Dict[int,float]] = None):
    import math, numpy as np

    state_dict_start = copy.deepcopy(model.state_dict())
    _bar(f"PASS {pass_idx} — {conv_path} (bn={bn_path}, pool={pool_path})", width=70)
    _ensure_gate_for_conv_path(model, conv_path, device)

    sw.go(f"mid_eval_s_L{pass_idx}")
    acc_layer0, per_layer0_list = eval_top1_and_perclass(model, loaders["eval"], device)
    sw.stop()
    layer0_per = {cid: acc for (cid, acc, _) in per_layer0_list}

    sw.go(f"gap_collect_s_L{pass_idx}")
    X_gap = collect_gap_matrix(model, loaders["gap"], device, pool_path, max_images=int(getattr(T, "SIM_BATCH_LIMIT", 8000)))
    sw.stop()

    metric  = str(getattr(T, "SIM_METRIC", "pearson")).lower()
    linkage = str(getattr(T, "LINKAGE", "centroid")).lower()
    sw.go(f"proximity_build_s_L{pass_idx}")
    prox = HCProximity(X_gap, metric=metric, linkage=linkage, max_init_pairs=getattr(T, "PROX_MAX_INIT_PAIRS", None))
    sw.stop(); _print_distance_stats_from_heap(prox)

    bs_eval = int(getattr(T, "BATCH_SIZE_EVAL", 512))
    if active_idx is None and bool(getattr(T, "ACTIVE_SET_ENABLE", True)):
        strict = bool(getattr(T, "ACTIVE_STRICT_EQUAL", True))
        act = pick_active_indices_stratified(
            model, X_eval, y_eval, device,
            max_total=int(getattr(T, "ACTIVE_MAX_SAMPLES", 4096)),
            per_class=int(getattr(T, "ACTIVE_PER_CLASS", 512)),
            bs=bs_eval, strict_equal=strict
        )
        active_idx = act
    Xe = X_eval[active_idx] if (active_idx is not None and active_idx.numel() > 0) else X_eval
    Ye = y_eval[active_idx] if (active_idx is not None and active_idx.numel() > 0) else y_eval

    S_set = set(prox.live)
    sw.go(f"calibration_s_L{pass_idx}")
    m_floor = calibrate_m_floor(model, conv_path, S_set, Xe, Ye, device, bs_eval, RATIO_MIN_BASE)
    sw.stop()
    print(f"[Setup] Calibration (benefit magnitude): m_floor={m_floor:.6f}")
    min_abs_flips_req = max(2, int(m_floor * Xe.size(0)))

    floor = int(getattr(T, "LAYER_SURVIVOR_FLOOR", {}).get(conv_path, 0))
    S0 = len(S_set); M_total = max(1, S0 - floor)
    tail_start_at = int(math.ceil((1.0 - TAIL_LAST_FRAC) * M_total))

    merges: List[dict] = []
    S_history: List[set] = [set(S_set)]
    steps = 0
    MAX_SCAN = int(getattr(T, "MAX_PAIRS_PROBED_PER_STEP", 256))
    announced_tail = False
    gate = _get_gate_by_conv_path(model, conv_path)
    C_layer = gate.mask.shape[1]

    # --- MAIN PRUNING LOOP ---
    sw.go(f"hc_run_s_L{pass_idx}")
    while True:
        if len(S_set) <= floor:
            print(f"[STOP][L{pass_idx}] floor reached: |S|={len(S_set)} <= {floor}"); break
        scanned = 0; merged_this_step = False

        # --- SAFETY BRAKE ---
        if SAFETY_BRAKE_ENABLE and steps > 0 and steps % SAFETY_BRAKE_INTERVAL == 0:
            dropped_base = sorted(set(range(C_layer)) - S_set)
            dropped_vec = torch.as_tensor(dropped_base, dtype=torch.long, device=device)
            gate.mask.index_fill_(1, dropped_vec, 0.0)
            acc_now = quick_acc_on_cache(model, X_eval, y_eval, device, MID_EVAL_SAMPLE, bs_eval)
            drop = acc_layer0 - acc_now
            gate.mask.fill_(1.0)
            if drop > SAFETY_BRAKE_ACC_DROP:
                print(f"🛑 [Safety Brake] Stop early. Drop {drop:.4f} > {SAFETY_BRAKE_ACC_DROP}")
                break

        while scanned < MAX_SCAN:
            best = prox.pop_best()
            if best is None: break
            i, j, sim_val = best; scanned += 1
            if (MIN_SIM_STOP > -1.0) and (sim_val < MIN_SIM_STOP):
                scanned = MAX_SCAN; break

            merges_done = S0 - len(S_set); tail_mode = merges_done >= tail_start_at
            ratio_eff = min(1.0, RATIO_MIN_BASE + (RATIO_MIN_TAIL_DELTA if tail_mode else 0.0))

            # -----------------------------------------------------
            # TIER 3 CHECK: Try Drop (Benefit Pair Early)
            # -----------------------------------------------------
            n01, n10, usedN = benefit_pair_early(model, conv_path, S_set, i, j, Xe, Ye, device, batch_size=bs_eval, ratio_min=ratio_eff, min_abs_flips_override=min_abs_flips_req)
            ok_drop, met_drop, _ = gate_decision_ratio_only(n01, n10, usedN, ratio_min=ratio_eff)

            if ok_drop:
                # --- TIER 3 ACCEPTANCE ---
                bi, bj = met_drop["bi"], met_drop["bj"]
                if abs(bi - bj) > 1e-12: keep, drop = (i, j) if bi >= bj else (j, i)
                else:
                    W = get_conv_module(model, conv_path).weight.detach().cpu()
                    norms = W.view(W.shape[0], -1).pow(2).sum(1).sqrt()
                    drop = i if norms[i] < norms[j] else j
                    keep = j if drop == i else i

                # Standard update for Drop
                prox.update_after_merge(keep=keep, drop=drop, op="avg")
                if drop in S_set: S_set.remove(drop)
                merges.append({"i": int(i), "j": int(j), "keep": int(keep), "drop": int(drop),
                               "sim": float(sim_val), "tier": 3, "custom_weight": None})
                S_history.append(set(S_set))
                merged_this_step = True; steps += 1
                break

            # -----------------------------------------------------
            # TIER 2 CHECK: Try Feature Map Crossover
            # -----------------------------------------------------
            elif getattr(T, "TIER2_ENABLE", True):
                # 1. Capture Feature Maps
                prof.inc("tier2_attempts")
                fm_data = {}
                conv_mod = get_conv_module(model, conv_path)

                # Hook to capture input to this layer
                input_capture = []
                def hook_in(mod, inp, out):
                    input_capture.append(inp[0].detach()) # [B, Cin, H, W]
                h_in = conv_mod.register_forward_hook(hook_in)

                # Run calibration batch
                # Only need enough to form a target, reuse Xe
                tier2_bs = min(64, Xe.size(0))
                xb = Xe[:tier2_bs].to(device)

                # We need the output of the parents i and j.
                # Since we hooked the INPUT, we can compute parent outputs manually
                # to avoid hooking internals if possible.
                try:
                    _ = model.features(xb) if hasattr(model, "features") else model(xb)
                except Exception: pass # Just need the hook to fire
                h_in.remove()

                if not input_capture: continue # Failed to capture

                X_in = input_capture[0] # [B, Cin, H, W]

                # Compute Parent FMs manually
                # F_i, F_j: [1, Cin, kH, kW]
                w_i = conv_mod.weight[i].unsqueeze(0)
                w_j = conv_mod.weight[j].unsqueeze(0)

                # Conv args
                pad, strd = conv_mod.padding, conv_mod.stride

                fm_i = F.conv2d(X_in, w_i, bias=None, stride=strd, padding=pad)
                fm_j = F.conv2d(X_in, w_j, bias=None, stride=strd, padding=pad)

                # 2. Generate Hybrid Target
                fm_hybrid = generate_hybrid_fm(fm_i.squeeze(1), fm_j.squeeze(1)) # [B, H, W]

                # --- CRITICAL FIX: Detach before regression ---
                fm_hybrid = fm_hybrid.detach()
                # ----------------------------------------------

                # 3. Regress New Weights
                t_reg = time.perf_counter()
                f_new = regress_filter_weights(model, conv_path, fm_hybrid, i, j, X_in)
                prof.add_time("tier2_regress_s", time.perf_counter() - t_reg)

                # 4. Validation (Swap & Check)
                # We need to temporarily implant f_new at index i, and mask out index j
                # to see if the network survives with just the hybrid.

                # Back up original weight
                orig_w_i = conv_mod.weight.data[i].clone()
                orig_mask = gate.mask.clone()

                # Implant
                conv_mod.weight.data[i] = f_new.squeeze(0)

                # Mask out J (simulate drop)
                mask_temp = gate.mask.clone()
                mask_temp[0, j, 0, 0] = 0.0
                gate.mask = mask_temp

                # Check accuracy
                acc_t2 = quick_acc_on_cache(model, X_eval, y_eval, device, MID_EVAL_SAMPLE, bs_eval)
                drop_t2 = acc_layer0 - acc_t2

                # Restore
                conv_mod.weight.data[i] = orig_w_i
                gate.mask = orig_mask

                # Decision
                # Use slightly looser bound for T2? No, use same ratio or direct drop check.
                # Since we measured actual drop, compare against ALLOWED drop per step?
                # Using a heuristic: If drop is negligible (< 0.5%), accept.
                if drop_t2 <= 0.005:
                    # --- TIER 2 ACCEPTANCE ---
                    keep, drop = i, j # By convention, we put hybrid in 'keep' slot

                    # Update prox: Treat as average merge for similarity purposes
                    prox.update_after_merge(keep=keep, drop=drop, op="avg")
                    if drop in S_set: S_set.remove(drop)

                    # Store custom weight!
                    merges.append({"i": int(i), "j": int(j), "keep": int(keep), "drop": int(drop),
                                   "sim": float(sim_val), "tier": 2,
                                   "custom_weight": f_new.squeeze(0).cpu()})
                    S_history.append(set(S_set))
                    prof.inc("tier2_success")
                    merged_this_step = True; steps += 1
                    print(f"  [Tier 2] Crossover success! Drop={drop_t2:.4f}")
                    break

            # If Tier 2 also fails (or disabled), loop continues (Tier 1 = Skip)

        if not merged_this_step:
            break

    sw.stop()
    print(f"[HC] Outcome: accepted_merges={len(merges)} | survivors |S|={len(S_set)}")

    # 3. SAFETY NET LOOP (Commit & Check)
    retries_left = ROLLBACK_STEPS if INCREMENTAL_ROLLBACK else 0
    current_merges = list(merges)
    current_S = set(S_set)
    success = False
    soft_rollback_from = len(merges)
    soft_rollback_to = len(merges)
    MAX_LAYER_DROP = float(getattr(T, "MAX_LAYER_DROP", 0.008))

    while True:
        conv = get_conv_module(model, conv_path); lin1, _ = adapter.resolve_linear1()
        shapes_pre = layer_shapes_prepool(model, device, conv_path, pool_path, loaders["eval"])
        Hc0, Wc0 = shapes_pre["pool"][2], shapes_pre["pool"][3]
        macs_conv0 = conv_macs_per_image(conv, conv.in_channels, Hc0, Wc0); macs_lin10 = linear_macs_per_image(lin1)
        params_total0 = sum(p.numel() for p in model.parameters()); C0 = conv.out_channels

        # Apply Surgery
        kept_indices = sorted(current_S)
        sw.go(f"prune_surgery_s_L{pass_idx}")
        model = prune_conv_block_and_linear_(model, device, conv_path, bn_path, pool_path, kept_indices, current_merges, loaders["eval"], adapter)
        if hasattr(torch, "_dynamo"): torch._dynamo.reset()
        sw.stop()

        # Recalibrate
        if X_bn_fixed is not None:
            _sync(); sw.go(f"bn_full_recal_s_L{pass_idx}")
            gate = _get_gate_by_conv_path(model, conv_path)
            if gate.mask.shape[1] != len(kept_indices):
                 gate.mask = torch.ones(1, len(kept_indices), 1, 1, device=device)
            bn_recalibrate_from_cache(model, X_bn_fixed, iters=int(getattr(T, "BN_FULL_ITERS", 2048)), device=device)
            sw.stop(); _sync()

        # Final Eval
        sw.go(f"mid_eval_s_L{pass_idx}"); acc_mid, per_mid = eval_top1_and_perclass(model, loaders["eval"], device); sw.stop()

        # Check Guards
        MAX_OVERALL = float(getattr(T, "GLOBAL_MAX_OVERALL_DROP", 0.02))
        MAX_CLASS   = float(getattr(T, "GLOBAL_MAX_CLASS_DROP",   0.06))
        baseline_ref = global_baseline_acc if global_baseline_acc is not None else acc_layer0
        drop_vs_ref = baseline_ref - acc_mid

        fail_msg = None
        if drop_vs_ref > MAX_OVERALL:
             fail_msg = f"Drop {drop_vs_ref:.4f} > {MAX_OVERALL}"

        drop_vs_start = acc_layer0 - acc_mid
        if fail_msg is None and drop_vs_start > MAX_LAYER_DROP:
             fail_msg = f"Layer Drop {drop_vs_start:.4f} > {MAX_LAYER_DROP:.4f}"

        if fail_msg is None and global_baseline_per is not None:
             per_mid_dict = {cid: acc for cid, acc, _ in per_mid}
             for cid, base_acc in global_baseline_per.items():
                 class_drop = base_acc - per_mid_dict.get(cid, base_acc)
                 if class_drop > MAX_CLASS:
                     fail_msg = f"Class {cid} Drop {class_drop:.4f} > {MAX_CLASS}"; break

        if fail_msg is None:
            success = True
            soft_rollback_to = len(current_merges)
            break

        if retries_left > 0:
            print(f"⚠️ [Retry] Layer attempt failed: {fail_msg}")
            _restore_model_structure(model, state_dict_start, getattr(T, "ALT_ORDER", []), device)
            model.load_state_dict(state_dict_start)
            n_merges = len(current_merges)
            n_keep = int(n_merges * (1.0 - ROLLBACK_RATIO))
            n_keep = max(0, min(n_keep, len(S_history) - 1))
            current_merges = current_merges[:n_keep]
            current_S = S_history[n_keep]
            retries_left -= 1
        else:
            print(f"❌ [Retry] All attempts failed. Reverting.")
            _restore_model_structure(model, state_dict_start, getattr(T, "ALT_ORDER", []), device)
            model.load_state_dict(state_dict_start)
            kept = C0; current_merges = []; soft_rollback_to = 0; kept_indices = list(range(C0)); success = False
            break

    conv_p = get_conv_module(model, conv_path); lin1_p, _ = adapter.resolve_linear1()
    macs_conv1 = conv_macs_per_image(conv_p, conv_p.in_channels, Hc0, Wc0); macs_lin11 = linear_macs_per_image(lin1_p)
    kept = len(kept_indices) if success else C0
    pruned_this = C0 - kept
    summary = {
        "conv": conv_path, "bn": bn_path, "pool": pool_path,
        "acc": acc_mid, "per": {cid: acc for (cid, acc, _) in per_mid},
        "kept": kept, "pruned": pruned_this, "merges": current_merges,
        "soft_rollback_from": soft_rollback_from, "soft_rollback_to": soft_rollback_to,
        "macs": {"conv0": int(macs_conv0), "conv1": int(macs_conv1), "lin10": int(macs_lin10), "lin11": int(macs_lin11)}
    }
    pre = (C0, macs_conv0, macs_lin10, params_total0)
    return model, summary, pre

In [ ]:
# ================================
# Phase 3 / Cell 4 — Orchestrator (Final: Global + Per-Class Awareness)
# ================================
import torch, torch.nn as nn, json, os, time
from typing import Dict, Any

def _make_eval_wrapper():
    base = lambda x: model(x)
    compiled = False
    if hasattr(torch, "compile") and torch.cuda.is_available() and bool(getattr(T, "ENABLE_COMPILE", True)):
        try:
            wrapped = torch.compile(base, mode="default", dynamic=True, fullgraph=False)
            compiled = True
        except Exception:
            wrapped = base
            compiled = False
    else:
        wrapped = base
    globals()["_EVAL_COMPILED"] = compiled
    return wrapped

def _human_model_name():
    if BACKBONE == "vgg16_bn":
        return "VGG16-BN"
    return getattr(model, "__class__", type("_", (), {})).__name__ or "Model"

def _print_alt_order():
    print(f"[B2] multi-layer pass enabled: {bool(getattr(T, 'B2_MULTI_LAYER', True))}")
    print("[B2] Alternating order (one-pass):")
    for k, L in enumerate(getattr(T, "ALT_ORDER", []), 1):
        quota = L.get("max_merges", getattr(T, "MAX_MERGES", None))
        qtxt  = "None" if quota in (None, "None") else str(quota)
        print(f"   {k}. conv={L['conv']} | bn={L.get('bn')} | pool={L['pool']} | quota={qtxt}")

def run_phase3_local(adapter: ModelAdapter, model: nn.Module, loaders: Dict[str,Any], device, T):
    timing = {}
    trajectory = []
    layer_summaries = []
    pre_stats = []

    def _capture_aggregate_stats() -> Dict[str, Any]:
        layer_specs = getattr(T, "ALT_ORDER", [])
        c_ch = 0
        c_params = 0
        c_macs = 0

        lin1, _ = adapter.resolve_linear1()
        l_params = count_params_module(lin1)
        l_macs = linear_macs_per_image(lin1)

        shape_cache = {}
        for spec in layer_specs:
            conv_path = spec["conv"]
            bn_path = spec.get("bn")
            pool_path = spec["pool"]

            conv = get_conv_module(model, conv_path)
            bn = try_get_bn_module(model, bn_path)

            if conv_path not in shape_cache:
                shapes = layer_shapes_prepool(model, device, conv_path, pool_path, loaders["eval"])
                Hc = int(shapes["conv"][2])
                Wc = int(shapes["conv"][3])
                shape_cache[conv_path] = (Hc, Wc)
            else:
                Hc, Wc = shape_cache[conv_path]

            c_ch += int(conv.out_channels)
            c_params += count_params_module(conv)
            if bn is not None:
                c_params += count_params_module(bn)
            c_macs += conv_macs_per_image(conv, conv.in_channels, Hc, Wc)

        params_total = sum(p.numel() for p in model.parameters())

        return {
            "channels_total": int(c_ch),
            "conv_params": int(c_params),
            "linear1_params": int(l_params),
            "conv_macs_per_image": float(c_macs),
            "linear1_macs_per_image": float(l_macs),
            "params_total": int(params_total),
        }

    for L in getattr(T, "ALT_ORDER", []):
        _ensure_gate_for_conv_path(model, L["conv"], device)
    globals()["FWD_EVAL"] = _make_eval_wrapper()

    print("✅ torch.compile enabled for EVAL (forward wrapper)" if globals().get("_EVAL_COMPILED", False) else "⚠️ torch.compile disabled for EVAL (forward wrapper)")
    _print_alt_order()

    _sync(); sw.go("baseline_eval_s")
    acc0, per0 = eval_top1_and_perclass(model, loaders["eval"], device)
    sw.stop(); _sync()
    timing["baseline_eval_s"] = sw.t["baseline_eval_s"]
    print(f"[hb] after baseline | +{timing['baseline_eval_s']:.1f}s elapsed")
    baseline_per = {cid: acc for (cid, acc, _) in per0}
    trajectory.append({"stage": "Baseline", "acc": acc0, "per": baseline_per})

    last_acc = acc0
    last_per = dict(baseline_per)

    sw.go("eval_cache_s")
    X_eval, y_eval = build_eval_cache(loaders["eval"], device, max_images=int(getattr(T, "EVAL_BATCH_LIMIT", len(loaders["eval"].dataset))))
    sw.stop(); timing["eval_cache_s"] = sw.t["eval_cache_s"]

    X_bn_fixed = None
    if HAS_BN:
        X_bn_fixed, _ = build_bn_calib_cache_from_eval(X_eval, y_eval, total=int(getattr(T, "BN_CALIB_TOTAL", 2048)))

    active_idx = None
    if bool(getattr(T, "ACTIVE_SET_ENABLE", True)):
        bs = int(getattr(T, "BATCH_SIZE_EVAL", 512))
        act = pick_active_indices_stratified(
            model, X_eval, y_eval, device,
            max_total=int(getattr(T, "ACTIVE_MAX_SAMPLES", 4096)),
            per_class=int(getattr(T, "ACTIVE_PER_CLASS", 512)),
            bs=bs, strict_equal=bool(getattr(T, "ACTIVE_STRICT_EQUAL", True)))
        if act.numel() < int(getattr(T, "ACTIVE_MIN_SAMPLES", 1024)):
            act = pick_active_indices_strict_equal(
                model, X_eval, y_eval, device,
                per_class=max(1, int(getattr(T, "ACTIVE_MIN_SAMPLES", 1024)) // max(1, len(torch.unique(y_eval)))),
                bs=bs)
        active_idx = act
        print(f"[active] N_eff={int(active_idx.numel())} (stratified={bool(getattr(T, 'ACTIVE_STRICT_EQUAL', True))}, refresh_every={int(getattr(T, 'ACTIVE_REFRESH_EVERY', 150))})")

    baseline_agg = _capture_aggregate_stats()

    for pass_idx, L in enumerate(getattr(T, "ALT_ORDER", []), 1):
        model_snapshot = copy.deepcopy(model)

        # Uses run_local_layer from Cell 3!
        model_after, summary, pre = run_local_layer(
            model, pass_idx,
            conv_path=L["conv"], bn_path=L.get("bn"), pool_path=L["pool"],
            X_eval=X_eval, y_eval=y_eval, X_bn_fixed=X_bn_fixed, active_idx=active_idx,
            prof=prof, sw=sw, device=device, loaders=loaders,
            adapter=adapter,
            verbose=False,
            max_merges=L.get("max_merges", getattr(T, "MAX_MERGES", None)),
            global_baseline_acc=acc0,
            global_baseline_per=baseline_per
        )

        # Redundant Global Guard (Sanity Check - should rarely fire now)
        MAX_OVERALL = float(getattr(T, "GLOBAL_MAX_OVERALL_DROP", 0.02))
        MAX_CLASS   = float(getattr(T, "GLOBAL_MAX_CLASS_DROP",   0.06))

        acc_pass = float(summary.get("acc", last_acc))
        per_pass = dict(summary.get("per", last_per))

        overall_drop = acc0 - acc_pass
        class_drops = {}
        for cid, acc0_c in baseline_per.items():
            acc_c = per_pass.get(cid, acc0_c)
            class_drops[cid] = acc0_c - acc_c
        max_class_drop = max(class_drops.values()) if class_drops else 0.0

        violates_overall = overall_drop > MAX_OVERALL
        violates_class   = max_class_drop > MAX_CLASS

        if violates_overall or violates_class:
            print("\n[Global Guard] Layer exceeded global rails even after internal soft-rollback; Hard Revert.")
            print(f"  Layer: {L['conv']}")
            print(f"  Overall drop vs baseline: {overall_drop:.4f} (max {MAX_OVERALL:.4f})")
            print(f"  Max per-class drop vs baseline: {max_class_drop:.4f} (max {MAX_CLASS:.4f})")

            model = model_snapshot
            adapter.model = model
            globals()["model"] = model

            summary["global_guard_violation"] = True
            summary["overall_drop_vs_baseline"] = float(overall_drop)
            summary["max_class_drop_vs_baseline"] = float(max_class_drop)

            trajectory.append({
                "stage": f"Rejected {L['conv']} (global guard)",
                "acc": last_acc,
                "per": dict(last_per),
            })
            continue

        summary["global_guard_violation"] = False
        summary["overall_drop_vs_baseline"] = float(overall_drop)
        summary["max_class_drop_vs_baseline"] = float(max_class_drop)

        model = model_after
        adapter.model = model

        params_total1 = sum(p.numel() for p in model.parameters())
        summary["params_total1"] = int(params_total1)

        globals()["model"] = model
        pre_stats.append(pre)
        layer_summaries.append(("L" + str(pass_idx), summary))

        trajectory.append({"stage": f"After {L['conv']}", "acc": acc_pass, "per": per_pass})
        last_acc = acc_pass
        last_per = per_pass

    if HAS_BN and X_bn_fixed is not None:
        sw.go("bn_recalibrate_s"); _sync()
        bn_recalibrate_from_cache(model, X_bn_fixed, iters=int(getattr(T, "BN_FULL_ITERS", 2048)), device=device)
        sw.stop(); _sync(); timing["bn_recalibrate_s"] = sw.t["bn_recalibrate_s"]

    stripped = 0
    for L in getattr(T, "ALT_ORDER", []):
        if strip_channel_gate_for_infer(model, L["conv"]):
            stripped += 1
    globals()["FWD_EVAL"] = _make_eval_wrapper()

    _sync(); sw.go("post_eval_s")
    acc1, per1 = eval_top1_and_perclass(model, loaders["eval"], device)
    sw.stop(); _sync(); timing["post_eval_s"] = sw.t["post_eval_s"]

    hc_cost = dict(prof.counters)
    final_agg = _capture_aggregate_stats()

    results = {
        "baseline": {"acc": acc0, "per": {cid: acc for (cid, acc, _) in per0}},
        "final": {"acc": acc1, "per": {cid: acc for (cid, acc, _) in per1}},
        "layer_summaries": layer_summaries,
        "trajectory": trajectory,
        "timing": timing,
        "gates_stripped": stripped,
        "pre_stats": pre_stats,
        "dataset_size": len(loaders["eval"].dataset) if hasattr(loaders["eval"], "dataset") else None,
        "model_name": _human_model_name(),
        "backbone": BACKBONE,
        "compiled_eval": bool(globals().get("_EVAL_COMPILED", False)),
        "hc_cost": hc_cost,
        "baseline_agg": baseline_agg,
        "final_agg": final_agg,
    }
    return results

# --- run it
run_summary = run_phase3_local(adapter, model, LOADERS, DEVICE, T)
print("✅ Phase-3 run complete.")


✅ torch.compile enabled for EVAL (forward wrapper)
[B2] multi-layer pass enabled: True
[B2] Alternating order (one-pass):
   1. conv=features.16 | bn=features.17 | pool=features.19 | quota=None
   2. conv=features.12 | bn=features.13 | pool=features.15 | quota=None
   3. conv=features.8 | bn=features.9 | pool=features.11 | quota=None
   4. conv=features.4 | bn=features.5 | pool=features.7 | quota=None
   5. conv=features.0 | bn=features.1 | pool=features.3 | quota=None


[Eval Top1]: 100%|██████████| 19867/19867 [00:58<00:00, 340.25it/s]


[hb] after baseline | +58.4s elapsed


[Build Eval Cache]: 100%|██████████| 4000/4000 [00:10<00:00, 377.19it/s]


[active] N_eff=1024 (stratified=True, refresh_every=150)

══════════════════════════════════════════════════════════════════════
PASS 1 — features.16 (bn=features.17, pool=features.19)
══════════════════════════════════════════════════════════════════════


...


══════════════════════════════════════════════════════════════════════
PASS 5 — features.0 (bn=features.1, pool=features.3)
══════════════════════════════════════════════════════════════════════


[Eval Top1]: 100%|██████████| 19867/19867 [00:56<00:00, 350.31it/s]
[Collect GAP]: 6016it [00:15, 386.80it/s]
[Proximity Build]: 100%|██████████| 2016/2016 [00:00<00:00, 79595.95it/s]


[prox] distance stats (1 - sim) over sampled pairs:
  mean=0.9361  std=0.6219  median=1.0358  p90=1.7037  p99=1.7303
  current top-sim=0.9996 (top-dist=0.0004)
[Setup] Calibration (benefit magnitude): m_floor=0.121094
  [Tier 2] Crossover success! Drop=-0.0189
  [Tier 2] Crossover success! Drop=-0.0109
  [Tier 2] Crossover success! Drop=-0.0109
  [Tier 2] Crossover success! Drop=-0.0192
[STOP][L5] floor reached: |S|=60 <= 60
[HC] Outcome: accepted_merges=4 | survivors |S|=60
[surgery] internal: kept=60 at features.0; sliced fan-in of features.4 to Cin=60


[Eval Top1]: 100%|██████████| 19867/19867 [00:53<00:00, 369.05it/s]


✅ Phase-3 run complete.


In [ ]:
# ================================
# Phase 3 / Cell 5 — Reporting & Artifacts (Parity+Deep)
# ================================
import os, json, csv
import torch
import numpy as np  # Added explicit import

# where to save (reuse Phase-2 CONFIG paths if present)
OUT_DIR = CONFIG.get("SAVE_DIR", "./output/phase3")
os.makedirs(OUT_DIR, exist_ok=True)

# -------- helpers --------
_model_tag = run_summary.get("model_name", "Model")
_valN = int(run_summary.get("dataset_size") or len(LOADERS["eval"].dataset))
_compiled_flag = "✅" if run_summary.get("compiled_eval", False) else "❌"

# pretty prints — Baseline block (cnn-only parity)
_bar("BASELINE — BEFORE PRUNING", width=70)
print(f"Model                                  {_model_tag} (eval compiled: {_compiled_flag})")
print(f"Dataset (VAL size)                     {_fmt_big(_valN)}")
print(f"Top-1 Accuracy                         {_fmt_pct(run_summary['baseline']['acc'])}")
_rule(width=70)
print("Per-class Accuracy:")
for cid, acc in sorted(run_summary["baseline"]["per"].items()):
    print(f"  • class {cid}: {_fmt_pct(acc)}")
_rule(width=70)

# PASS-by-PASS detailed parity
baseline_acc = float(run_summary["baseline"]["acc"])
baseline_per = dict(run_summary["baseline"]["per"])

C_total0 = 0
cum_pruned = 0

for (tag, layer), pre in zip(run_summary["layer_summaries"], run_summary.get("pre_stats", [])):
    conv_path = layer["conv"]
    kept = int(layer["kept"]); pruned = int(layer["pruned"])
    acc_mid = float(layer["acc"])
    per_mid = layer["per"]
    mac0, mac1 = int(layer["macs"]["conv0"]), int(layer["macs"]["conv1"])
    lin0, lin1 = int(layer["macs"]["lin10"]), int(layer["macs"]["lin11"])
    C0, params0 = int(pre[0]), int(pre[3])
    params1 = int(layer.get("params_total1", params0))

    # cumulative channel stats
    C_total0 += C0
    cum_pruned += pruned

    # Count Tiers
    tiers = [m.get("tier", 3) for m in layer["merges"]]
    t2_count = sum(1 for t in tiers if t == 2)
    t3_count = sum(1 for t in tiers if t == 3)

    _bar(f"PASS {tag.split('L')[-1]} — {conv_path}", width=70)
    print(f"[Eval Top1]: {_fmt_pct(acc_mid)}")
    print(f"[Setup] GAP: X_gap.shape=(≈{CONFIG.get('PRUNE_REF_SAMPLES', getattr(T,'PRUNE_REF_SAMPLES',6000))}, {C0})")
    print(f"Kept/Pruned: {kept} / {pruned}")
    print(f"Merges accepted: {len(layer['merges'])} (Tier 3 Drop: {t3_count}, Tier 2 Hybrid: {t2_count})")

    # CHECKPOINT like cnn-only
    _bar(f"{tag} — CHECKPOINT", width=70)
    d_vs_base = acc_mid - baseline_acc
    print(f"Accuracy (Top-1)               {_fmt_pct(acc_mid)}  (Δ vs base: {_fmt_pp(d_vs_base)})")
    _rule(width=70)
    print("Per-class Accuracy (Δ vs base):")
    print(f"Class       Acc           Δ vs base     ")
    _rule(width=40)
    for cid in sorted(per_mid.keys()):
        acc_c = per_mid[cid]
        dvb_c = acc_c - baseline_per.get(cid, acc_c)
        print(f"class {cid:<2}    {_fmt_pct(acc_c):<12}  {_fmt_pp(dvb_c):<12}")

    # Pruning (this layer)
    print("\nPruning (this layer)")
    print("Metric          Before → After                Δ             %         ")
    _rule(width=70)
    print(f"OutCh           {C0} → {kept:<24} {-(C0-kept):<13} {_fmt_pct((C0-kept)/max(1,C0))}")
    print(f"Conv FLOPs* {_fmt_big(mac0)} → {_fmt_big(mac1):<12} {_fmt_big(mac1-mac0):<13}")
    print(f"Linear1 FLOPs* {_fmt_big(lin0)} → {_fmt_big(lin1):<12} {_fmt_big(lin1-lin0):<13}")
    print(f"Params total    {_fmt_big(params0)} → {_fmt_big(params1):<12} {_fmt_big(params1-params0):<13}")

    # Cumulative
    print("\nCumulative (up to", tag, ")")
    print("Metric        Count                       %         ")
    _rule(width=70)
    pct = (cum_pruned / max(1, C_total0))
    print(f"Channels      {cum_pruned} / {C_total0:<24} {_fmt_pct(pct)}")
    _rule(width=70)

# FINAL
_bar("FINAL — AFTER PRUNING", width=70)
print(f"Top-1 Accuracy: {_fmt_pct(run_summary['final']['acc'])}")
print("Per-class:")
for cid, acc in sorted(run_summary["final"]["per"].items()):
    print(f"  • class {cid}: {_fmt_pct(acc)}")
_rule(width=70)

# ACCURACY TRAJECTORY
print("ACCURACY TRAJECTORY")
print("Stage           Top-1       Δ vs Base     Δ vs Prev   ")
_rule(width=60)
prev = None
for step in run_summary["trajectory"]:
    s = step["stage"]
    a = float(step["acc"])
    dvb = a - baseline_acc
    dvp = (a - prev) if (prev is not None) else None
    print(f"{s:<15} {_fmt_pct(a):<10} {(_fmt_pp(dvb)):<13} {( _fmt_pp(dvp) if dvp is not None else '—'):<10}")
    prev = a
_rule(width=70)

# TIMING SUMMARY (cnn-only parity keys)
print("TIMING")
for k, v in run_summary["timing"].items():
    print(f"  {k}: {v:.2f}")
_total = sum(float(v) for v in sw.t.values()) if isinstance(sw.t, dict) else sum(run_summary["timing"].values())
print(f"  total_s: {_total:.2f}")
_rule(width=70)

# PERF SUMMARY
lat0 = run_summary["timing"].get("baseline_eval_s", 0.0) / max(1, _valN)
lat1 = run_summary["timing"].get("post_eval_s", 0.0) / max(1, _valN)
opt_overhead = _total - run_summary["timing"].get("baseline_eval_s", 0.0) - run_summary["timing"].get("post_eval_s", 0.0)
print("PERF SUMMARY")
print(f"Latency per sample             {lat0*1000:.3f} ms → {lat1*1000:.3f} ms ({(lat0/max(1e-9,lat1)):.2f}×)")
print(f"Optimization overhead (s)      {opt_overhead:.2f}")
print("FLOPs* note                    per-image MACs proxy")
_rule(width=70)

# HC COST PROFILE
if run_summary.get("hc_cost"):
    print("HC COST PROFILE")
    cc = run_summary["hc_cost"]
    for k in ["heap_pops","heap_pushes","pairs_sim_recomputed","benefit_evals","benefit_eval_s","sim_update_s", "tier2_attempts", "tier2_success"]:
        if k in cc:
            print(f"  {k}: {cc[k]}")
    _rule(width=70)

# Unified experiment summary (JSON-friendly schema)
try:
    import os as _os, json as _json

    # Latency stats (already computed above)
    lat0_ms = lat0 * 1000.0
    lat1_ms = lat1 * 1000.0

    # Convenience aliases from run_summary
    lb = run_summary["baseline"]
    lf = run_summary["final"]
    b_agg = run_summary.get("baseline_agg", {})
    f_agg = run_summary.get("final_agg", {})
    timing = run_summary.get("timing", {})

    _baseline_per = dict(lb.get("per", {}))
    _final_per = dict(lf.get("per", {}))

    # --- Meta / config / guards (using only active rails) ------------------
    _meta = {
        "approach": "local",
        "backbone": BACKBONE,
        "dataset": CONFIG.get("DATASET", "celeba").lower() if isinstance(CONFIG.get("DATASET", "celeba"), str) else "celeba",
    }

    _config_summary = {
        "img_size": list(CONFIG.get("IMG_SIZE", (224, 224))),
        "num_classes": int(CONFIG.get("NUM_CLASSES", 2)),
        "train_samples": int(CONFIG.get("TRAIN_SAMPLES", 10000)),
        "prune_ref_samples": int(CONFIG.get("PRUNE_REF_SAMPLES", 6000)),
        "eval_samples": int(_valN),
        "batch_size_train": int(CONFIG.get("BATCH_SIZE", 64)),
        "batch_size_eval": int(getattr(T, "BATCH_SIZE_EVAL", CONFIG.get("BATCH_SIZE_EVAL", 512))),
        "seed": int(CONFIG.get("SEED", 42)),
    }

    _guards_summary = {
        # Step-wise guard: HC safety brake (overall accuracy drop vs layer baseline)
        "step_acc_max": float(SAFETY_BRAKE_ACC_DROP),
        "step_class_rel_max": None,
        # Cumulative rails vs global baseline and per-class drops
        "cum_acc_max": float(getattr(T, "GLOBAL_MAX_OVERALL_DROP", 0.02)),
        "cum_class_rel_max": float(getattr(T, "GLOBAL_MAX_CLASS_DROP", 0.06)),
        # Drift-related knobs are no-ops in S5 core:
        "perclass_drift_thresh": None,
        "perclass_consec_n": None,
    }

    # --- Baseline / final core metrics -------------------------------------
    _c_ch0   = int(b_agg.get("channels_total", 0))
    _c_params0 = int(b_agg.get("conv_params", 0))
    _l_params0 = int(b_agg.get("linear1_params", 0))
    _c_macs0   = float(b_agg.get("conv_macs_per_image", 0.0))
    _l_macs0   = float(b_agg.get("linear1_macs_per_image", 0.0))
    _params_total0 = int(b_agg.get("params_total", 0))

    params_total1 = int(f_agg.get("params_total", 0))
    c_params1     = int(f_agg.get("conv_params", 0))
    l_params1     = int(f_agg.get("linear1_params", 0))
    c_macs1       = float(f_agg.get("conv_macs_per_image", 0.0))
    l_macs1       = float(f_agg.get("linear1_macs_per_image", 0.0))

    _baseline = {
        "acc_top1": float(lb["acc"]),
        "per_class_acc": _baseline_per,
        "channels_per_layer": {},  # filled below
        "params_total": int(_params_total0),
        "params_conv": int(_c_params0),
        "params_linear1": int(_l_params0),
        "macs_conv_per_image": float(_c_macs0),
        "macs_linear1_per_image": float(_l_macs0),
        "flops_conv_per_image": float(_c_macs0 * 2.0),
        "flops_linear1_per_image": float(_l_macs0 * 2.0),
        "latency_ms_per_sample": float(lat0_ms),
        "eval_time_s": float(timing.get("baseline_eval_s", 0.0)),
    }

    _final = {
        "acc_top1": float(lf["acc"]),
        "per_class_acc": _final_per,
        "channels_per_layer": {},  # filled below
        "params_total": int(params_total1),
        "params_conv": int(c_params1),
        "params_linear1": int(l_params1),
        "macs_conv_per_image": float(c_macs1),
        "macs_linear1_per_image": float(l_macs1),
        "flops_conv_per_image": float(c_macs1 * 2.0),
        "flops_linear1_per_image": float(l_macs1 * 2.0),
        "latency_ms_per_sample": float(lat1_ms),
        "eval_time_s": float(timing.get("post_eval_s", 0.0)),
    }

    # --- Channels per layer (before/after) ---------------------------------
    layer_specs = getattr(T, "ALT_ORDER", [])

    _baseline_channels = {}
    for (tag, layer), pre in zip(run_summary["layer_summaries"], run_summary.get("pre_stats", [])):
        conv_path = layer["conv"]
        C0 = int(pre[0])
        _baseline_channels[conv_path] = C0

    _channels_after = {
        spec["conv"]: int(get_conv_module(model, spec["conv"]).out_channels)
        for spec in layer_specs
    }

    _baseline_channels_total = int(sum(_baseline_channels.values()))
    _final_channels_total    = int(sum(_channels_after.values()))

    _baseline["channels_per_layer"] = dict(_baseline_channels)
    _final["channels_per_layer"]    = dict(_channels_after)

    # --- Compression summary -----------------------------------------------
    _acc_drop = _baseline["acc_top1"] - _final["acc_top1"]
    _per_class_drops = {
        cid: _baseline_per.get(cid, 0.0) - _final_per.get(cid, 0.0)
        for cid in set(list(_baseline_per.keys()) + list(_final_per.keys()))
    }
    _max_class_drop = max(_per_class_drops.values()) if _per_class_drops else 0.0

    _compression = {
        "acc_drop_overall_pp": float(_acc_drop * 100.0),
        "max_per_class_drop_pp": float(_max_class_drop * 100.0),
        "channels_total_before": _baseline_channels_total,
        "channels_total_after": _final_channels_total,
        "channels_total_pruned": _baseline_channels_total - _final_channels_total,
        "channels_total_pruned_pct": float(
            (_baseline_channels_total - _final_channels_total) / max(1, _baseline_channels_total)
        ),
        "params_total_before": _baseline["params_total"],
        "params_total_after": _final["params_total"],
        "params_total_pruned": _final["params_total"] - _baseline["params_total"],
        "params_total_pruned_pct": float(
            (_baseline["params_total"] - _final["params_total"]) / max(1, _baseline["params_total"])
        ),
        "conv_params_before": _baseline["params_conv"],
        "conv_params_after": _final["params_conv"],
        "conv_params_pruned": _final["params_conv"] - _baseline["params_conv"],
        "linear1_params_before": _baseline["params_linear1"],
        "linear1_params_after": _final["params_linear1"],
        "linear1_params_pruned": _final["params_linear1"] - _baseline["params_linear1"],
        "macs_conv_before": _baseline["macs_conv_per_image"],
        "macs_conv_after": _final["macs_conv_per_image"],
        "macs_linear1_before": _baseline["macs_linear1_per_image"],
        "macs_linear1_after": _final["macs_linear1_per_image"],
    }

    # --- Layer-wise summary -------------------------------------------------
    _layers_summary = {}
    for conv_path, _b_ch in _baseline_channels.items():
        _a_ch = int(_channels_after.get(conv_path, _b_ch))
        _layers_summary[conv_path] = {
            "channels_before": _b_ch,
            "channels_after": _a_ch,
            "channels_pruned": _b_ch - _a_ch,
            "channels_pruned_pct": float((_b_ch - _a_ch) / max(1, _b_ch)),
        }

    # --- Timing & ratios ----------------------------------------------------
    _timing_summary = dict(timing)
    _optimization_overhead = float(
        opt_overhead  # computed above from total_s - baseline_eval_s - post_eval_s
    )

    _ratios = {
        "param_ratio_total": float(_final["params_total"] / max(1, _baseline["params_total"])),
        "param_ratio_conv": float(_final["params_conv"] / max(1, _baseline["params_conv"])),
        "param_ratio_linear1": float(_final["params_linear1"] / max(1, _baseline["params_linear1"])),
        "mac_ratio_conv": float(_final["macs_conv_per_image"] / max(1e-9, _baseline["macs_conv_per_image"])),
        "mac_ratio_linear1": float(_final["macs_linear1_per_image"] / max(1e-9, _baseline["macs_linear1_per_image"])),
        "latency_speedup": float(lat0_ms / lat1_ms) if lat1_ms > 0 else float("inf"),
        "overhead_vs_eval": float(
            _optimization_overhead / max(1e-9, timing.get("baseline_eval_s", 0.0))
        ),
    }

    _compiled_flag = bool(globals().get("_EVAL_COMPILED", run_summary.get("compiled_eval", False)))
    _notes = {
        "compiled_eval": _compiled_flag,
        "gates_stripped": int(run_summary.get("gates_stripped", 0)),
        "hc_cost": run_summary.get("hc_cost", {}),
    }

    GLOBAL_EXPERIMENT_SUMMARY = {
        "meta": _meta,
        "config": _config_summary,
        "guards": _guards_summary,
        "baseline": _baseline,
        "final": _final,
        "compression": _compression,
        "layers": _layers_summary,
        "timing": _timing_summary,
        "ratios": _ratios,
        "notes": _notes,
    }

    # Also embed into run_summary for convenience
    run_summary["experiment_summary"] = GLOBAL_EXPERIMENT_SUMMARY

    # Persist unified summary alongside phase3 JSON (separate file)
    _out_dir = CONFIG.get("SAVE_DIR", OUT_DIR)
    _os.makedirs(_out_dir, exist_ok=True)
    _fname = f"phase3_summary_local_{_meta['backbone']}_{_meta['dataset']}.json"
    _summary_path = _os.path.join(_out_dir, _fname)
    with open(_summary_path, "w") as _f:
        _json.dump(GLOBAL_EXPERIMENT_SUMMARY, _f, indent=2)
    print(f"💾 Saved unified experiment summary → {_summary_path}")
except Exception as _e:
    print(f"⚠️ Failed to build local experiment summary: {_e}")

# --- JSON SANITIZATION ---
def _sanitize_for_json(data):
    if isinstance(data, dict):
        return {k: _sanitize_for_json(v) for k, v in data.items() if k != "custom_weight"}
    elif isinstance(data, (list, tuple, set)): # Fixed: handle tuple/set
        return [_sanitize_for_json(v) for v in data]
    elif isinstance(data, (torch.Tensor, np.ndarray)):
        return str(data.shape) # Replace heavy tensor with its shape
    return data

# save JSON (Safe Dump)
run_summary_safe = _sanitize_for_json(run_summary)
json_path = os.path.join(OUT_DIR, "phase3_summary.json")
with open(json_path, "w") as f:
    json.dump(run_summary_safe, f, indent=2)
print(f"💾 Saved JSON → {json_path}")

# save merges per layer as CSVs
for tag, layer in run_summary["layer_summaries"]:
    csv_path = os.path.join(OUT_DIR, f"{tag}_merges.csv")
    with open(csv_path, "w", newline="") as f:
        # Added "tier" to the fieldnames
        w = csv.DictWriter(f, fieldnames=["i","j","keep","drop","sim","r","bi","bj","N", "tier"])
        w.writeheader()
        for row in layer["merges"]:
            # Drop the heavy custom_weight from CSV dictionary before writing
            clean_row = {k: v for k, v in row.items() if k != "custom_weight"}
            # Ensure tier exists (default 3 for old logs)
            if "tier" not in clean_row: clean_row["tier"] = 3
            w.writerow(clean_row)
    print(f"💾 Saved merges → {csv_path}")


══════════════════════════════════════════════════════════════════════
BASELINE — BEFORE PRUNING
══════════════════════════════════════════════════════════════════════
Model                                  CNN_V1 (eval compiled: ✅)
Dataset (VAL size)                     19,867
Top-1 Accuracy                         94.15%
──────────────────────────────────────────────────────────────────────
Per-class Accuracy:
  • class 0: 93.86%
  • class 1: 94.53%
──────────────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════
PASS 1 — features.16
══════════════════════════════════════════════════════════════════════
[Eval Top1]: 93.28%
[Setup] GAP: X_gap.shape=(≈6000, 512)
Kept/Pruned: 160 / 352
Merges accepted: 352 (Tier 3 Drop: 207, Tier 2 Hybrid: 145)

══════════════════════════════════════════════════════════════════════
L1 — CHECKPOINT
══════════════════════════════════════════════════════════════════════
Accuracy